# 05 – TFA


Constraint-based analysis on genome-scale metabolic models (GEMs) is a popular method to study metabolism and cellular physiology. Flux Balance Analysis (FBA), in particular, has been used to predict network-level behaviors, such as specific growth rate, gene essentiality, etc. However, FBA-derived approaches often lead to flux distributions
that are contradicting with physiology and bioenergetics due to the
lack of thermodynamic constraints in their formulation. Metabolic reactions in GEMs can be futher constrained thermodynamically using TFA (Salvy...Meric et., al, 2019).

References

*   https://github.com/EPFL-LCSB/pytfa/tree/master
*   Bioinformatics, 35(1), 2019, 167–169
doi: 10.1093/bioinformatics/bty499



# Install dependencies

In [1]:
!pip3 install cobra pytfa catboost optlang modelseedpy


Requested pytfa from https://files.pythonhosted.org/packages/bb/ac/1d6a4a72f45bfa46a95d3cc9f29c113c5f4a4ece4ac7461b56ac61c2fa2c/pytfa-0.9.4-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    python-version (>="3.6") ; extra == 'equilibrator'
                   ~^
Please use pip<24.1 if you need to use this version.
Requested pytfa from https://files.pythonhosted.org/packages/a3/b4/9d3a72b34043fc5e4b75afcefd2d386ebab18ca99690cfa1c4c01612c77d/pytfa-0.9.3-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    python-version (>="3.6") ; extra == 'equilibrator'
                   ~^
Please use pip<24.1 if you need to use this version.
Requested pytfa from https://files.pythonhosted.org/packages/52/75/1b2114796a7f0b51ce973798805a5ccf5e929078bc65e59dc44b4f5601ea/pytfa-0.9.2-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PAR

In [ ]:
# Check installation
!pip3 show cobra

Name: cobra
Version: 0.30.0
Summary: COBRApy is a package for constraint-based modeling of metabolic networks.
Home-page: https://opencobra.github.io/cobrapy
Author: The cobrapy core development team.
Author-email: cobra-pie@googlegroups.com
License: LGPL-2.0-or-later OR GPL-2.0-or-later
Location: /usr/local/lib/python3.12/dist-packages
Requires: appdirs, depinfo, diskcache, future, httpx, numpy, optlang, pandas, pydantic, python-libsbml, rich, ruamel.yaml, swiglpk
Required-by: ModelSEEDpy, pytfa


In [ ]:
!pip3 show pytfa

Name: pytfa
Version: 0.9.1
Summary: pyTFA, Thermodynamics-based Flux Analysis in Python
Home-page: https://github.com/EPFL-LCSB/pytfa/
Author: pyTFA team
Author-email: softwares.lcsb@epfl.ch
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: bokeh, cobra, networkx, optlang, pytest, scipy, tqdm
Required-by: 


# Mount drive

In [ ]:
import os
from pathlib import Path
from google.colab import drive

def mount_drive():
  drive.mount('/content/drive', force_remount=True)
  drive_folder = "metabolic_modelling/phb-optimization-rpalustris/"
  os.chdir('/content/drive/MyDrive/'+ drive_folder)
  global PROJECT_ROOT
  PROJECT_ROOT = Path(os.getcwd())

mount_drive()


Mounted at /content/drive


# Load packages

In [ ]:
# Load packages

import logging
import cobra
from cobra import Reaction, Metabolite, Model, io
from cobra.io import (
    load_model, load_json_model, save_json_model,
    load_matlab_model, save_matlab_model,
    read_sbml_model, write_sbml_model
)

from cobra.flux_analysis import flux_variability_analysis, pfba

from cobra.util.solver import linear_reaction_coefficients

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
from itertools import product
import seaborn as sns
import json
import math
import copy
from tqdm import tqdm
import sys
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import pytfa, pkgutil, inspect
from pytfa import ThermoModel
from pytfa.io import load_thermoDB
from pytfa.optim.variables import DeltaG
from pytfa.analysis import variability_analysis

from pytfa.io import read_lexicon, annotate_from_lexicon, read_compartment_data, apply_compartment_data


# Paths

In [ ]:
# Import path saved in src

import sys
sys.path.append(str(PROJECT_ROOT / "src"))
from src.paths import PHB_TFA_DIR, PHB_MODEL_TFA_DIR, DATA_DIR, MODELS_DIR, PHB_MODEL_DIR, PHB_CHECKPOINTS_DIR, PHB_RESULTS_DIR, PHB_FIGURES_DIR, PHB_GEM_EXPERIMENTAL_DIR, PHB_GEM_AUGMENTATION_DIR, PHB_CATBOOST_DIR, PHB_PARETO_DIR


# Load thermo data




In [ ]:
# Download thermo database
!wget https://raw.githubusercontent.com/EPFL-LCSB/pytfa/master/data/thermo_data.thermodb -O thermo_data.thermodb

# Load thermodynamics

thermo_data_path = DATA_DIR / "thermo_data.thermodb"
thermo_data = load_thermoDB(thermo_data_path)

# Print thermo_data keys

print(f"Thermo Data loaded successfully:{thermo_data.keys()}")

--2026-03-08 18:28:42--  https://raw.githubusercontent.com/EPFL-LCSB/pytfa/master/data/thermo_data.thermodb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2184378 (2.1M) [application/octet-stream]
Saving to: ‘thermo_data.thermodb’

thermo_data.thermod 100%[===================>]   2.08M  --.-KB/s    in 0.1s    

2026-03-08 18:28:42 (18.9 MB/s) - ‘thermo_data.thermodb’ saved [2184378/2184378]

Thermo Data loaded successfully:dict_keys(['name', 'units', 'metabolites', 'cues'])


# *R. palustris GEM

Constrained model in previous notebooks (02_model_rpalustris_PHB_constrained.xml)

In [ ]:
# Load models

# Full path to the SBML file
model_path = PHB_MODEL_DIR / "02_model_rpalustris_PHB_constrained.xml"

# Alternatively use model associated to the Pareto Optimal solution with the maximum PHB production
# model_path = PHB_MODEL_DIR / ""best_pareto_PHB_model.xml"

# Load model
model_phb = io.read_sbml_model(
    str(model_path),
    use_fbc=False
)

print("Model loaded successfully")



Model loaded successfully


In [ ]:
model_phb.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ac_e,EX_ac_e,0.2477,2,100.00%
photon690_e,EX_photon690_e,0.626,0,0.00%
phbg_c,SK_phbg_c,0.1238,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
PHB_c,DM_PHB_c,-0.1238,4,99.92%
h2o_e,EX_h2o_e,-5E-05,0,0.00%
o2_e,EX_o2_e,-0.1238,0,0.00%
2obut_c,sink_2obut_c,-0.0001,4,0.08%


# Check PHB synthesis reaction values


In [ ]:
rxn = model_phb.reactions.get_by_id('PHBS_syn')
print("PHBS_syn bounds:", rxn.bounds)
print("PHBS_syn stoichiometry:", rxn.reaction)

PHBS_syn bounds: (0.0, 0.3978)
PHBS_syn stoichiometry: 3hbcoa__R_c + phbg_c --> PHB_c + coa_c


# Check Photon absorption reaction values


In [ ]:
rxn = model_phb.reactions.get_by_id('EX_photon690_e')
print("EX_photon690_e bounds:", rxn.bounds)
print("EX_photon690_e stoichiometry:", rxn.reaction)

EX_photon690_e bounds: (-1000.0, 1000.0)
EX_photon690_e stoichiometry: photon690_e <=> 


# Load lexicon and compartment data  



In [ ]:
# Load lexicon
lexicon_path = PHB_MODEL_TFA_DIR / "lexicon_iDT1294.csv" # load lexicon with 8 reactions

# Load compartments
compartments_path = PHB_MODEL_TFA_DIR / "compartment_data.json" # "compartments_data_iDT1294_test.json"

# Copy 02_model_rpalustris_PHB_constrained model
model_constrained = model_phb.copy()


# *Correct key formulas in model

Some formulas are incorrect. Here, only a handful of formulas of key reactions are corrected.

In [ ]:
import re
from collections import Counter

# --------------------------------
# INPUT DATA
# --------------------------------
Thermodb_meta_interest = [
  "C21H27N7O17P3",
  "C21H27N7O14P2",
  "C23H35N7O17P3S",
  "C21H26N7O14P2",
  "C25H39N7O18P3S",
  "C21H26N7O17P3",
  "C25H37N7O18P3S",
  "C21H33N7O16P3S"
]

GEM_meta_interest = [
  "C21H26N7O17P3",
  "C21H27N7O14P2",
  "C23H34N7O17P3S",
  "C21H26N7O14P2",
  "C25H38N7O18P3S",
  "C21H25N7O17P3",
  "C25H36N7O18P3S",
  "C21H32N7O16P3S"
]

# GEM metabolite IDs in SAME ORDER
gem_met_ids = [
  "nadph_c",
  "nadh_c",
  "accoa_c",
  "nad_c",
  "3hbcoa__R_c",
  "nadp_c",
  "aacoa_c",
  "coa_c",
]

# --------------------------------
# FORMULA PARSING UTILITIES
# --------------------------------
def parse_formula(formula):
  """Convert formula string to Counter"""
  return Counter({
      elem: int(count) if count else 1
      for elem, count in re.findall(r'([A-Z][a-z]?)(\d*)', formula)
  })

def counter_to_formula(counter):
  """Convert Counter back to formula string (Hill order, omit 1s)"""
  parts = []

  # Hill system: C, H, then others alphabetically
  if 'C' in counter:
      parts.append(f"C{counter['C']}" if counter['C'] != 1 else "C")
  if 'H' in counter:
      parts.append(f"H{counter['H']}" if counter['H'] != 1 else "H")

  for el in sorted(counter):
      if el in ('C', 'H'):
          continue
      count = counter[el]
      parts.append(f"{el}{count}" if count != 1 else el)

  return ''.join(parts)


def compare_formulas(thermo_f, gem_f):
  """Return element-wise difference GEM − ThermoDB"""
  t = parse_formula(thermo_f)
  g = parse_formula(gem_f)
  elements = set(t) | set(g)
  return {el: g.get(el, 0) - t.get(el, 0) for el in elements}

def update_gem_formula(gem_f, thermo_f):
  """Add missing elements from ThermoDB into GEM formula"""
  g = parse_formula(gem_f)
  t = parse_formula(thermo_f)

  for el, count in t.items():
      if g.get(el, 0) < count:
          g[el] = count

  return counter_to_formula(g)

# --------------------------------
# APPLY UPDATES TO MODEL
# --------------------------------
for met_id, thermo_f, gem_f in zip(gem_met_ids,
                                  Thermodb_meta_interest,
                                  GEM_meta_interest):

    diff = compare_formulas(thermo_f, gem_f)

    # Safety check (CRITICAL for TFA)
    for el in diff:
        if el in ("C", "N", "P") and diff[el] != 0:
            raise ValueError(
                f"Unsafe formula mismatch for {met_id}: element {el}"
            )

    new_formula = update_gem_formula(gem_f, thermo_f)

    if new_formula != gem_f:
        met = model_constrained.metabolites.get_by_id(met_id) ## update cobra model

        print(f"Updating {met_id}")
        print(f"  old: {gem_f}")
        print(f"  new: {new_formula}")
        print(f"  ThermoDB:       {thermo_f}")

        met.formula = new_formula

        # PRINT FROM MODEL OBJECT
        print(f"  new (model):  {met.formula}")
        print("-" * 40)


print("✅ GEM formulas updated and TFA-ready.")

Updating nadph_c
  old: C21H26N7O17P3
  new: C21H27N7O17P3
  ThermoDB:       C21H27N7O17P3
  new (model):  C21H27N7O17P3
----------------------------------------
Updating accoa_c
  old: C23H34N7O17P3S
  new: C23H35N7O17P3S
  ThermoDB:       C23H35N7O17P3S
  new (model):  C23H35N7O17P3S
----------------------------------------
Updating 3hbcoa__R_c
  old: C25H38N7O18P3S
  new: C25H39N7O18P3S
  ThermoDB:       C25H39N7O18P3S
  new (model):  C25H39N7O18P3S
----------------------------------------
Updating nadp_c
  old: C21H25N7O17P3
  new: C21H26N7O17P3
  ThermoDB:       C21H26N7O17P3
  new (model):  C21H26N7O17P3
----------------------------------------
Updating aacoa_c
  old: C25H36N7O18P3S
  new: C25H37N7O18P3S
  ThermoDB:       C25H37N7O18P3S
  new (model):  C25H37N7O18P3S
----------------------------------------
Updating coa_c
  old: C21H32N7O16P3S
  new: C21H33N7O16P3S
  ThermoDB:       C21H33N7O16P3S
  new (model):  C21H33N7O16P3S
----------------------------------------
✅ GEM formu

# Opt: Load medium constraints from checkpoint
 Use if best Pareto model conditions are not loaded properly

In [ ]:
# Define function to apply medium condition from excel.
def apply_media_from_excel(model, excel_path, close_exchanges=True):
    """
    Apply media constraints from a CSV or Excel file with columns:
    Reaction | LowerBound | UpperBound
    """

    excel_path = Path(excel_path)

    # --- Read file safely ---
    if excel_path.suffix.lower() == ".csv":
        media_df = pd.read_csv(excel_path)
    else:
        media_df = pd.read_excel(excel_path)

    # --- Clean reaction IDs ---
    media_df["Reaction"] = media_df["Reaction"].astype(str).str.strip()

    required_cols = {"Reaction", "LowerBound", "UpperBound"}
    if not required_cols.issubset(media_df.columns):
        raise ValueError(f"File must contain columns: {required_cols}")

    # --- Close uptake only ---
    if close_exchanges:
        for rxn in model.exchanges:
            rxn.lower_bound = 0.0

    # --- Apply media ---
    for _, row in media_df.iterrows():
        rxn_id = row["Reaction"]

        if rxn_id not in model.reactions:
            print(f"Reaction {rxn_id} not found — skipping")
            continue

        rxn = model.reactions.get_by_id(rxn_id)
        rxn.lower_bound = float(row["LowerBound"])
        rxn.upper_bound = float(row["UpperBound"])

    # --- Debug ---
    ex_ac = model.reactions.get_by_id("EX_ac_e")
    print("DEBUG → EX_ac_e bounds:", ex_ac.bounds)

    return model


In [ ]:
# Load medium conditions from Pareto results
project_root = Path(os.getcwd())
excel_path_pareto_medium = PHB_CHECKPOINTS_DIR / "phb_model_medium_montiel_buitron_2022_exchange_bounds_right_before_fba.csv"

# Print file
media_df = pd.read_csv(excel_path_pareto_medium)
media_df

/content/driveDL/MyDrive/Genome_scale_metabolic_models_PNSB/Model_Tec_Campos_2023_RPiDT1294/rpalustris_pha_optimization/models/phb/checkpoint_output/phb_model_medium_montiel_buitron_2022_exchange_bounds_right_before_fba.csv


In [ ]:
#model_constrained = model_constrained.copy()
model_constrained = apply_media_from_excel(
    model_constrained,
    excel_path_pareto_medium,
    close_exchanges=True
)

DEBUG → EX_ac_e bounds: (-0.2477, 1000.0)


# *Add compartments to cobra model

In [ ]:
model_constrained.compartments = {
        'c': 'Cytosol',
        'e': 'Extracellular',
        'p': 'Periplasm',
        'u': 'Unknown',
    }

In [ ]:
model_constrained.compartments = {
        'c': {
            'pH': 7.0,
            'ionicStr': 0.25,
            'c_min': 1e-6,
            'c_max': 0.02,
            'membranePot': 0.0,
        },
        'e': {
            'pH': 7.0,
            'ionicStr': 0.25,
            'c_min': 1e-6,
            'c_max': 0.02,
            'membranePot': 0.0,
        },
        'p': {
            'pH': 7.0,
            'ionicStr': 0.25,
            'c_min': 1e-6,
            'c_max': 0.02,
            'membranePot': 0.15,   # periplasm vs cytosol (example)
        },
        'u': {
            'pH': 7.0,
            'ionicStr': 0.25,
            'c_min': 1e-6,
            'c_max': 0.02,
            'membranePot': 0.0,
        }
    }

# Checkup function for wide value reactions

In [ ]:
# Check that all reaction bounds are <>1000

too_wide = [
    rxn.id for rxn in model_constrained.reactions
    if rxn.upper_bound > 1000 or rxn.lower_bound < -1000
]

print(f"Reactions still exceeding ±1000 bounds: {len(too_wide)}")


Reactions still exceeding ±1000 bounds: 0


# Multiple points TFA V4.1

Include operational and calculated parameters, fba, tfba, and their reaction fluxes.
This version does not handle data from diffferent CatBoost modelling strategies (i.e., "_con", "_sup", "_con_pfba"). For that purpose see the next version.


In [ ]:

# ============================================================
# USER PARAMETERS
# ============================================================
logging.getLogger("thermomodel_None").setLevel(logging.ERROR)
logging.getLogger("pytfa").setLevel(logging.ERROR)

PHB_SYN_UB_VALUES = [0.3978]#, 1000

# ============================================================
# CONSTANTS
# ============================================================
PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1",
]

SUB_TO_RXN = {
    "ace": "EX_ac_e",
    "lac": "EX_lac__L_e",
    "ppa": "EX_ppa_e",
    "but": "EX_but_e",
    "ibt": "EX_ibt_e",
    "mal": "EX_mal__L_e",
    "hxa": "EX_hxa_e",
    "oct": "EX_octa_e",
    "nh4": "EX_nh4_e",
    "hco3": "EX_hco3_e",
}

EXCHANGE_RXNS = list(SUB_TO_RXN.values())

# ============================================================
# SOLVER SETTINGS
# ============================================================
def apply_solver_settings(model):
    model.solver = "glpk"
    model.solver.configuration.tolerances.feasibility = 1e-9
    model.solver.configuration.presolve = True


# ============================================================
# LOAD TFA METADATA (ONCE)
# ============================================================
lexicon = read_lexicon(str(lexicon_path))
comp_data = read_compartment_data(str(compartments_path))


# ============================================================
# MAIN LOOP
# ============================================================
for PHB_SYN_UB in PHB_SYN_UB_VALUES:

    print(f"\n==============================")
    print(f"PHB_SYN_UB_VALUE: {PHB_SYN_UB}")
    print(f"==============================")

    PHB_SYN_TAG = f"{PHB_SYN_UB:g}"

    INPUT_CSV = PHB_PARETO_DIR / f"03_pareto_best_all_strategies_{PHB_SYN_TAG}.csv"
    OUT_CSV = PHB_TFA_DIR / f"05_FBA_TFA_PHB_detailed_results_{PHB_SYN_TAG}.csv"

    df = pd.read_csv(INPUT_CSV)

    if len(df) != 5:
        print(
            f" Warning: expected 3 Pareto solutions, found {len(df)} "
            f"for PHB_SYN_UB = {PHB_SYN_UB}"
        )

    results = []

    # ============================================================
    # LOOP OVER PARETO SOLUTIONS
    # ============================================================
    for pareto_id, row in df.iterrows():

        print(f"\n--- Pareto solution {pareto_id} ---")

        # -------------------------
        # BUILD MODEL
        # -------------------------
        m = model_constrained.copy()

        # -------------------------
        # APPLY EXCHANGE CONSTRAINTS
        # -------------------------
        for col, rxn_id in SUB_TO_RXN.items():
            if col in row and rxn_id in m.reactions:
                val = row[col]
                if pd.notna(val) and float(val) != 0.0:
                    rxn = m.reactions.get_by_id(rxn_id)
                    rxn.lower_bound = -abs(float(val))  # uptake
                    rxn.upper_bound = 0.0

        # -------------------------
        # OBJECTIVE: PHB
        # -------------------------
        phb_rxn = m.reactions.get_by_id("PHBS_syn")
        phb_rxn.lower_bound = 0.0
        phb_rxn.upper_bound = PHB_SYN_UB
        m.objective = phb_rxn

        # -------------------------
        # NO GROWTH
        # -------------------------
        biomass = m.reactions.get_by_id("BIOMASS__1")
        biomass.lower_bound = 0.0
        biomass.upper_bound = 0.0

        apply_solver_settings(m)

        # -------------------------
        # FBA
        # -------------------------
        fba_sol = m.optimize()
        pfba_sol = pfba(m)

        # -------------------------
        # TFA
        # -------------------------
        mytfa = pytfa.ThermoModel(thermo_data, m.copy())
        annotate_from_lexicon(mytfa, lexicon)
        apply_compartment_data(mytfa, comp_data)
        mytfa.objective = "PHBS_syn"
        apply_solver_settings(mytfa)
        mytfa.prepare()

        tfa_sol = mytfa.optimize()

        # -------------------------
        # COLLECT RESULTS
        # -------------------------
        result = {
            "PHB_SYN_UB": PHB_SYN_UB,
            "PHB_SYN_TAG": PHB_SYN_TAG,
            "pareto_id": pareto_id,
            "PHB_pFBA": pfba_sol.fluxes.get("PHBS_syn", np.nan),
            "PHB_FBA": fba_sol.fluxes.get("PHBS_syn", np.nan),
            "PHB_TFA": tfa_sol.fluxes.get("PHBS_syn", np.nan),
            "Biomass_pFBA": pfba_sol.fluxes.get("BIOMASS__1", np.nan),
            "Biomass_FBA": fba_sol.fluxes.get("BIOMASS__1", np.nan),
            "Biomass_TFA": tfa_sol.fluxes.get("BIOMASS__1", np.nan),
            "status_pFBA": pfba_sol.status,
            "status_FBA": fba_sol.status,
            "status_TFA": tfa_sol.status,
        }

        for r in PHB_REACTIONS:
            result[f"{r}_pFBA"] = pfba_sol.fluxes.get(r, np.nan)
            result[f"{r}_FBA"] = fba_sol.fluxes.get(r, np.nan)
            result[f"{r}_TFA"] = tfa_sol.fluxes.get(r, np.nan)

        for r in EXCHANGE_RXNS:
            result[f"{r}_pFBA"] = pfba_sol.fluxes.get(r, np.nan)
            result[f"{r}_FBA"] = fba_sol.fluxes.get(r, np.nan)
            result[f"{r}_TFA"] = tfa_sol.fluxes.get(r, np.nan)

        results.append(result)

    # ============================================================
    # SAVE RESULTS (3 ROWS PER PHB_SYN_UB)
    # ============================================================
    results_df = pd.DataFrame(results)

    results_df.to_csv(OUT_CSV, index=False)

    print(f"\nDONE — Results saved to:\n{OUT_CSV}")


Output hidden; open in https://colab.research.google.com to view.

# Multiple points TFA V4

It handles only one PHBS_syn value and no _con, _sup, etc

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import cobra
import pytfa

from pytfa.io import (
    read_lexicon,
    annotate_from_lexicon,
    read_compartment_data,
    apply_compartment_data,
)

# ============================================================
# USER INPUT
# ============================================================
INPUT_CSV = PHB_PARETO_DIR / "03_pareto_best_all_strategies_0.3978.csv"
OUT_CSV   = PHB_TFA_DIR / "03_FBA_TFA_PHB_detailed_results_20260308.csv"

# ============================================================
# CONSTANTS
# ============================================================
PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1",
]

# ============================================================
# COLUMN → EXCHANGE REACTION MAP
# ============================================================
SUB_TO_RXN = {
    "ace": "EX_ac_e",
    "lac": "EX_lac__L_e",
    "ppa": "EX_ppa_e",
    "but": "EX_but_e",
    "ibt": "EX_ibt_e",
    "mal": "EX_mal__L_e",
    "hxa": "EX_hxa_e",
    "oct": "EX_octa_e",
    "NH4_model_flux": "EX_nh4_e",
    "hco3": "EX_hco3_e",
}

EXCHANGE_RXNS = list(SUB_TO_RXN.values())

# ============================================================
# SOLVER SETTINGS
# ============================================================
def apply_solver_settings(model):
    model.solver = "glpk"
    model.solver.configuration.tolerances.feasibility = 1e-9
    model.solver.configuration.presolve = True

# ============================================================
# LOAD TFA METADATA (ONCE)
# ============================================================
lexicon   = read_lexicon(str(lexicon_path))
comp_data = read_compartment_data(str(compartments_path))

# ============================================================
# LOAD PARETO SOLUTIONS (MULTIPLE ROWS)
# ============================================================
df = pd.read_csv(INPUT_CSV)


# keep only first row
df = df.iloc[:1]


if df.empty:
    raise ValueError("Pareto CSV is empty")

print(f"Loaded {len(df)} Pareto-optimal solutions")

# ============================================================
# OUTPUT COLUMNS
# ============================================================
ALL_COLUMNS = (
    ["Pareto_ID", "PHB_FBA", "PHB_TFA",
     "Biomass_FBA", "Biomass_TFA",
     "status_FBA", "status_TFA"]
    + [f"{r}_FBA" for r in PHB_REACTIONS]
    + [f"{r}_TFA" for r in PHB_REACTIONS]
    + [f"{r}_FBA" for r in EXCHANGE_RXNS]
    + [f"{r}_TFA" for r in EXCHANGE_RXNS]
)

all_results = []

# ============================================================
# MAIN LOOP OVER PARETO SOLUTIONS
# ============================================================
for idx, row in df.iterrows():

    print(f"\n--- Processing Pareto solution {idx} ---")

    # -------------------------
    # BUILD FRESH MODEL
    # -------------------------
    m = model_constrained.copy()
    apply_solver_settings(m)

    # -------------------------
    # APPLY SUBSTRATE CONSTRAINTS
    # -------------------------
    for col, rxn_id in SUB_TO_RXN.items():
        if col in row and rxn_id in m.reactions:
            val = row[col]
            if pd.notna(val) and float(val) != 0.0:
                rxn = m.reactions.get_by_id(rxn_id)
                rxn.lower_bound = -abs(float(val))
                rxn.upper_bound = 0.0

    # -------------------------
    # NO GROWTH
    # -------------------------
    biomass = m.reactions.get_by_id("BIOMASS__1")
    biomass.lower_bound = 0.0
    biomass.upper_bound = 0.0

    # -------------------------
    # PHB OBJECTIVE
    # -------------------------
    phb_rxn = m.reactions.get_by_id("PHBS_syn")
    phb_rxn.lower_bound = 0.0
    phb_rxn.upper_bound = 0.3978
    m.objective = phb_rxn

    # ============================================================
    # FBA
    # ============================================================
    fba_sol = m.optimize()

    # ============================================================
    # TFA
    # ============================================================
    mytfa = pytfa.ThermoModel(thermo_data, m.copy())
    annotate_from_lexicon(mytfa, lexicon)
    apply_compartment_data(mytfa, comp_data)

    mytfa.objective = "PHBS_syn"
    apply_solver_settings(mytfa)
    mytfa.prepare()
    mytfa.convert()


    tfa_sol = mytfa.optimize()

    # ============================================================
    # COLLECT RESULTS
    # ============================================================
    result = {
        "Pareto_ID": idx,
        "PHB_FBA": fba_sol.fluxes.get("PHBS_syn", np.nan),
        "PHB_TFA": tfa_sol.fluxes.get("PHBS_syn", np.nan),
        "Biomass_FBA": fba_sol.fluxes.get("BIOMASS__1", np.nan),
        "Biomass_TFA": tfa_sol.fluxes.get("BIOMASS__1", np.nan),
        "status_FBA": fba_sol.status,
        "status_TFA": tfa_sol.status,
    }

    for r in PHB_REACTIONS:
        result[f"{r}_FBA"] = fba_sol.fluxes.get(r, np.nan)
        result[f"{r}_TFA"] = tfa_sol.fluxes.get(r, np.nan)

    for r in EXCHANGE_RXNS:
        result[f"{r}_FBA"] = fba_sol.fluxes.get(r, np.nan)
        result[f"{r}_TFA"] = tfa_sol.fluxes.get(r, np.nan)

    all_results.append(result)

# ============================================================
# SAVE RESULTS
# ============================================================
results_df = pd.DataFrame(all_results)[ALL_COLUMNS]
results_df.to_csv(OUT_CSV, index=False)

print("\nDONE — FBA vs TFA comparison saved.")
print(f"Results written to: {OUT_CSV}")


Loaded 1 Pareto-optimal solutions

--- Processing Pareto solution 0 ---


Streaming output truncated to the last 5000 lines.
2026-03-08 19:11:53,328 - thermomodel_None - WARNING - dtdpglu_c  not found in annotations
2026-03-08 19:11:53,329 - thermomodel_None - WARNING - 1odecg3p_c  not found in annotations
2026-03-08 19:11:53,331 - thermomodel_None - WARNING - 1odec11eg3p_c  not found in annotations
2026-03-08 19:11:53,332 - thermomodel_None - WARNING - octe_9_ACP_c  not found in annotations
2026-03-08 19:11:53,333 - thermomodel_None - WARNING - 1odec912eg3p_c  not found in annotations
2026-03-08 19:11:53,335 - thermomodel_None - WARNING - octe_9_12_ACP_c  not found in annotations
2026-03-08 19:11:53,337 - thermomodel_None - WARNING - 1odec691215eg3p_c  not found in annotations
2026-03-08 19:11:53,338 - thermomodel_None - WARNING - octe_6_9_12_15_ACP_c  not found in annotations
2026-03-08 19:11:53,340 - thermomodel_None - WARNING - 1odec91215eg3p_c  not found in annotations
2026-03-08 19:11:53,341 - thermomodel_None - WARNING - octe_9_12_15_ACP_c  not found 

DEBUG:thermomodel_None:SPMDabcpp : thermo constraint NOT created
DEBUG:thermomodel_None:DHAD2 : thermo constraint NOT created
DEBUG:thermomodel_None:PRAMPC : thermo constraint NOT created
DEBUG:thermomodel_None:3OAS80 : thermo constraint NOT created
DEBUG:thermomodel_None:G1PCTYT : thermo constraint NOT created
DEBUG:thermomodel_None:BIOMASS_CELL_WALL : thermo constraint NOT created
DEBUG:thermomodel_None:CA2abcpp : thermo constraint NOT created
DEBUG:thermomodel_None:UHGADA2 : thermo constraint NOT created
DEBUG:thermomodel_None:ASPO6 : thermo constraint NOT created
DEBUG:thermomodel_None:ASPK : thermo constraint NOT created
DEBUG:thermomodel_None:DXPS : thermo constraint NOT created
DEBUG:thermomodel_None:PRATPP : thermo constraint NOT created
DEBUG:thermomodel_None:NH4tpp : thermo constraint NOT created
DEBUG:thermomodel_None:PAPSR : thermo constraint NOT created
DEBUG:thermomodel_None:NAD_H2 : thermo constraint NOT created
DEBUG:thermomodel_None:CYRDAAT : thermo constraint NOT crea

DEBUG:thermomodel_None:3HAD40_2 : thermo constraint NOT created
DEBUG:thermomodel_None:EAR180y : thermo constraint NOT created
DEBUG:thermomodel_None:GTHRDH_syn : thermo constraint NOT created
DEBUG:thermomodel_None:PRAIi : thermo constraint NOT created
DEBUG:thermomodel_None:CYNTtabcpp : thermo constraint NOT created
DEBUG:thermomodel_None:PTPATi : thermo constraint NOT created
DEBUG:thermomodel_None:PC6AR_1 : thermo constraint NOT created
DEBUG:thermomodel_None:Kabcpp : thermo constraint NOT created
DEBUG:thermomodel_None:ENO : thermo constraint NOT created
DEBUG:thermomodel_None:PRFGS : thermo constraint NOT created
DEBUG:thermomodel_None:PC8XM : thermo constraint NOT created
DEBUG:thermomodel_None:3OAS60 : thermo constraint NOT created
DEBUG:thermomodel_None:GMPS2 : thermo constraint NOT created
DEBUG:thermomodel_None:RBFK : thermo constraint NOT created
DEBUG:thermomodel_None:NTRIRfx : thermo constraint NOT created
DEBUG:thermomodel_None:GTHPi : thermo constraint NOT created
DEBUG

DEBUG:thermomodel_None:EX_k_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_no3_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_fe3_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_mobd_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_ni2_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_na1_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_cynt_e : thermo constraint NOT created
DEBUG:thermomodel_None:EX_h_e : thermo constraint NOT created
DEBUG:thermomodel_None:FE3tex : thermo constraint NOT created
DEBUG:thermomodel_None:FE2tex : thermo constraint NOT created
DEBUG:thermomodel_None:NI2tex : thermo constraint NOT created
DEBUG:thermomodel_None:MNtex : thermo constraint NOT created
DEBUG:thermomodel_None:COBALT2tex : thermo constraint NOT created
DEBUG:thermomodel_None:CU2tex : thermo constraint NOT created
DEBUG:thermomodel_None:Zn2tex : thermo constraint NOT created
DEBUG:thermomodel_None:SPMDtex : thermo constraint NO

DEBUG:thermomodel_None:CYSS : thermo constraint NOT created
DEBUG:thermomodel_None:CYSTA : thermo constraint NOT created
DEBUG:thermomodel_None:CYTBDpp_1 : thermo constraint NOT created
DEBUG:thermomodel_None:CYTBDu : thermo constraint NOT created
DEBUG:thermomodel_None:DASYN160 : thermo constraint NOT created
DEBUG:thermomodel_None:DASYN161 : thermo constraint NOT created
DEBUG:thermomodel_None:DASYN180 : thermo constraint NOT created
DEBUG:thermomodel_None:DASYN181 : thermo constraint NOT created
DEBUG:thermomodel_None:DASYN181_9 : thermo constraint NOT created
DEBUG:thermomodel_None:DASYN182_9_12 : thermo constraint NOT created
DEBUG:thermomodel_None:DASYN183_6_9_12 : thermo constraint NOT created
DEBUG:thermomodel_None:DASYN183_9_12_15 : thermo constraint NOT created
DEBUG:thermomodel_None:DASYN184_6_9_12_15 : thermo constraint NOT created
DEBUG:thermomodel_None:DHDPS : thermo constraint NOT created
DEBUG:thermomodel_None:DHORDi : thermo constraint NOT created
DEBUG:thermomodel_Non

DEBUG:thermomodel_None:LIPAabcpp : thermo constraint NOT created
DEBUG:thermomodel_None:LIPAabctex : thermo constraint NOT created
DEBUG:thermomodel_None:LIPOS : thermo constraint NOT created
DEBUG:thermomodel_None:LKDRA : thermo constraint NOT created
DEBUG:thermomodel_None:LPADSS : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL1A120pp : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL1A140pp : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL1A141pp : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL1A160pp : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL1A161pp : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL1A180pp : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL1A181pp : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL1E120pp : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL1E140pp : thermo constraint NOT created
DEBUG:thermomodel_None:LPLIPAL1E141pp : thermo constrai

DEBUG:thermomodel_None:MMSAD2 : thermo constraint NOT created
DEBUG:thermomodel_None:MMSAD3 : thermo constraint NOT created
DEBUG:thermomodel_None:MNabc : thermo constraint NOT created
DEBUG:thermomodel_None:MOTH1 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTH2 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTH3 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTH4 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTS1 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTS2 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTS3 : thermo constraint NOT created
DEBUG:thermomodel_None:MOTS4 : thermo constraint NOT created
DEBUG:thermomodel_None:MTHFR2_1 : thermo constraint NOT created
DEBUG:thermomodel_None:MTI : thermo constraint NOT created
DEBUG:thermomodel_None:NADHDH : thermo constraint NOT created
DEBUG:thermomodel_None:NADPHQR2 : thermo constraint NOT created
DEBUG:thermomodel_None:NADPHQR3 : thermo constraint NOT created
DEBUG:thermomo

Streaming output truncated to the last 5000 lines.
DEBUG:thermomodel_None:Added constraint: UF_MDDCP1ex
DEBUG:thermomodel_None:Added constraint: UR_MDDCP1ex
DEBUG:thermomodel_None:generating only use constraints for reactionMDDCP4ex
DEBUG:thermomodel_None:Added variable: FU_MDDCP4ex
DEBUG:thermomodel_None:Added variable: BU_MDDCP4ex
DEBUG:thermomodel_None:Added constraint: SU_MDDCP4ex
DEBUG:thermomodel_None:Added constraint: UF_MDDCP4ex
DEBUG:thermomodel_None:Added constraint: UR_MDDCP4ex
DEBUG:thermomodel_None:generating only use constraints for reactionMDDCP5ex
DEBUG:thermomodel_None:Added variable: FU_MDDCP5ex
DEBUG:thermomodel_None:Added variable: BU_MDDCP5ex
DEBUG:thermomodel_None:Added constraint: SU_MDDCP5ex
DEBUG:thermomodel_None:Added constraint: UF_MDDCP5ex
DEBUG:thermomodel_None:Added constraint: UR_MDDCP5ex
DEBUG:thermomodel_None:generating only use constraints for reactionMETabc
DEBUG:thermomodel_None:Added variable: FU_METabc
DEBUG:thermomodel_None:Added variable: BU_METa

Total variables: 13010
Total constraints: 10290
Number of ΔG variables: 142
['CDGS', 'CDGS_reverse_6b7cb', 'CDGR', 'CDGR_reverse_e4464', 'SQDGS_PALM_PALM', 'SQDGS_PALM_PALM_reverse_eec5a', 'SQDGS_HDE_PALM', 'SQDGS_HDE_PALM_reverse_af549', 'DGDGS_HDE_PALM', 'DGDGS_HDE_PALM_reverse_d95be']
Binary variables: 5442
['FU_QULNS', 'BU_QULNS', 'FU_ORNDC', 'BU_ORNDC', 'FU_MSBENZMT', 'BU_MSBENZMT', 'FU_DESAT18a', 'BU_DESAT18a', 'FU_FUM', 'BU_FUM']
Metabolite concentration variables: 2125
['SQLC_reverse_77a98', 'LC_gam6p_c', 'LC_cgly_c', 'LC_achms_c', 'LC_pcox_u', 'LC_octe9ACP_c', 'LC_fdp_c', 'LC_5caiz_c', 'LC_pep_c', 'LC_coa_c']


In [ ]:
PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1"
]


print("\n---- Thermodynamic constraints for PHB reactions ----")

for r_id in PHB_REACTIONS:

    dg_var = f"DG_{r_id}"
    fu_var = f"FU_{r_id}"
    bu_var = f"BU_{r_id}"

    has_dg = dg_var in mytfa.variables
    has_binary = fu_var in mytfa.variables or bu_var in mytfa.variables

    print(
        f"{r_id:12s}",
        "✅ thermo" if has_dg else "❌ no thermo",
        "| binary:" if has_binary else "| no direction vars"
    )


In [ ]:
assert any(c.name.startswith("G_") for c in mytfa.constraints)
assert any(v.name.startswith("DGo_") for v in mytfa.variables)
assert any(v.type == "binary" for v in mytfa.variables)
print("✅ TFBA is active")

# *Multiple points TFA V4.2  

This version handles multiple PHBS_syn_UB and data CatBoost modelling strategies

In [ ]:
# ============================================================
# USER PARAMETERS
# ============================================================

logging.getLogger("thermomodel_None").setLevel(logging.ERROR)
logging.getLogger("pytfa").setLevel(logging.ERROR)

PHB_SYN_UB_VALUES =  [0.3978] #, 1000
# ============================================================
# CONSTANTS
# ============================================================
PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1",
]

SUB_TO_RXN_BASE = {
    "ace": "EX_ac_e",
    "lac": "EX_lac__L_e",
    "ppa": "EX_ppa_e",
    "but": "EX_but_e",
    "ibt": "EX_ibt_e",
    "mal": "EX_mal__L_e",
    "hxa": "EX_hxa_e",
    "oct": "EX_octa_e",
    "nh4": "EX_nh4_e",
    "hco3": "EX_hco3_e",
}

EXCHANGE_RXNS = list(SUB_TO_RXN_BASE.values())

# ============================================================
# SOLVER SETTINGS
# ============================================================
def apply_solver_settings(model):
    model.solver = "glpk"
    model.solver.configuration.tolerances.feasibility = 1e-9
    model.solver.configuration.presolve = True


# ============================================================
# LOAD TFA METADATA (ONCE)
# ============================================================
lexicon = read_lexicon(str(lexicon_path))
comp_data = read_compartment_data(str(compartments_path))


# ============================================================
# MAIN LOOP
# ============================================================
for PHB_SYN_UB in PHB_SYN_UB_VALUES:

    print("\n==============================")
    print(f"PHB_SYN_UB_VALUE: {PHB_SYN_UB}")
    print("==============================")

    PHB_SYN_TAG = f"{PHB_SYN_UB:g}"

    INPUT_CSV = PHB_PARETO_DIR / f"03_pareto_best_all_strategies_{PHB_SYN_TAG}.csv"
    OUT_CSV = PHB_TFA_DIR / f"05_FBA_TFA_PFBA_detailed_results_all_strategies_{PHB_SYN_TAG}.csv"

    df = pd.read_csv(INPUT_CSV)
    results = []

    for pareto_id, row in df.iterrows():

        strategy = row["optimization_strategy"]

        if "SUP" in strategy:
            suffix = "_sup"

        elif strategy == "CON":
            suffix = "_con"

        elif strategy == "SUP_and_CON":
            suffix = "_sup"

        elif strategy == "PFBA":
            suffix = "_con_pfba"

        elif strategy == "SUP_and_PFBA":
            suffix = "_sup"

        else:
            print('Other optimization strategy')

        m = model_constrained.copy()

        # ---------------------------------
        # STORE INPUT VALUES USED
        # ---------------------------------
        input_values_used = {}

        for base_col, rxn_id in SUB_TO_RXN_BASE.items():

            col_name = f"{base_col}{suffix}"
            val_used = np.nan

            if col_name in row:
                val = row[col_name]
                if pd.notna(val) and float(val) != 0.0 and rxn_id in m.reactions:
                    rxn = m.reactions.get_by_id(rxn_id)
                    rxn.lower_bound = -abs(float(val))
                    rxn.upper_bound = 0.0
                    val_used = float(val)

            input_values_used[f"{base_col}_supplied"] = 0.0 if pd.isna(val_used) else val_used

        # ---------------------------------
        # OBJECTIVE & NO GROWTH
        # ---------------------------------
        phb_rxn = m.reactions.get_by_id("PHBS_syn")
        phb_rxn.lower_bound = 0.0
        phb_rxn.upper_bound = PHB_SYN_UB
        m.objective = phb_rxn

        biomass = m.reactions.get_by_id("BIOMASS__1")
        biomass.lower_bound = 0.0
        biomass.upper_bound = 0.0

        apply_solver_settings(m)

        # ---------------------------------
        # FBA
        # ---------------------------------
        fba_sol = m.optimize()

        # ---------------------------------
        # pFBA
        # ---------------------------------

        pfba_sol = pfba(m)

        # ---------------------------------
        # TFA
        # ---------------------------------
        # Build ThermoModel
        mytfa = pytfa.ThermoModel(thermo_data, m.copy())

        # Apply annotations
        annotate_from_lexicon(mytfa, lexicon)
        apply_compartment_data(mytfa, comp_data)

        # Set objective
        mytfa.objective = "PHBS_syn"

        # Prepare thermodynamic data
        mytfa.prepare()

        # Convert model to thermodynamic formulation
        mytfa.convert()

        # Apply solver settings AFTER conversion
        apply_solver_settings(mytfa)

        # ---------------------------------
        # VERIFY THAT TFA IS ACTIVE
        # ---------------------------------
        from pytfa.optim.variables import DeltaG

        dg_vars = mytfa.get_variables_of_type(DeltaG)

        if len(dg_vars) > 0:
            print(f"  TFA ACTIVE — {len(dg_vars)} ΔG variables created for Pareto solution {pareto_id}")

            # Example: print ΔG bounds for a reaction
            try:
                rxn_aa = mytfa.reactions.get_by_id("AACOAR_syn")
                dg_var = next(v for v in dg_vars if v.reaction.id == rxn_aa.id)

                print(
                    f"    AACOAR_syn ΔG bounds: ({dg_var.lb:.2f}, {dg_var.ub:.2f}) kJ/mol"
                )

            except Exception:
                print("    AACOAR_syn ΔG variable not found.")

        else:
            print(
                f"  TFA NOT ACTIVE for Pareto solution {pareto_id} "
                "(no ΔG variables created → model behaves like FBA)"
            )

        # ---------------------------------
        # OPTIMIZE TFA
        # ---------------------------------
        try:
            tfa_sol = mytfa.optimize()

        except Exception as e:
            print(f"TFA optimization failed for Pareto solution {pareto_id}: {e}")
            tfa_sol = fba_sol

        # ---------------------------------
        # COLLECT RESULTS
        # ---------------------------------
        result = {
            "PHB_SYN_UB": PHB_SYN_UB,
            "PHB_SYN_TAG": PHB_SYN_TAG,
            "pareto_id": pareto_id,
            "PHB_pFBA": pfba_sol.fluxes.get("PHBS_syn", np.nan),
            "PHB_FBA": fba_sol.fluxes.get("PHBS_syn", np.nan),
            "PHB_TFA": tfa_sol.fluxes.get("PHBS_syn", np.nan),
            "Biomass_pFBA": pfba_sol.fluxes.get("BIOMASS__1", np.nan),
            "Biomass_FBA": fba_sol.fluxes.get("BIOMASS__1", np.nan),
            "Biomass_TFA": tfa_sol.fluxes.get("BIOMASS__1", np.nan),
            "status_pFBA": pfba_sol.status,
            "status_FBA": fba_sol.status,
            "status_TFA": tfa_sol.status,
            "optimization_strategy": strategy
        }

        for r in PHB_REACTIONS:
            result[f"{r}_pFBA"] = pfba_sol.fluxes.get(r, np.nan)
            result[f"{r}_FBA"] = fba_sol.fluxes.get(r, np.nan)
            result[f"{r}_TFA"] = tfa_sol.fluxes.get(r, np.nan)

        for base_col, rxn_id in SUB_TO_RXN_BASE.items():
            result[f"{base_col}_pFBA"] = pfba_sol.fluxes.get(rxn_id, np.nan)
            result[f"{base_col}_FBA"]  = fba_sol.fluxes.get(rxn_id, np.nan)
            result[f"{base_col}_TFA"]  = tfa_sol.fluxes.get(rxn_id, np.nan)

        result.update(input_values_used)
        print(pd.Series(result))
        print("-" * 60)
        results.append(result)

    pd.DataFrame(results).to_csv(OUT_CSV, index=False)
    print(f"\nDONE — Results saved to:\n{OUT_CSV}")



PHB_SYN_UB_VALUE: 0.3978
  TFA ACTIVE — 1 ΔG variables created for Pareto solution 0
    AACOAR_syn ΔG variable not found.
PHB_SYN_UB         0.3978
PHB_SYN_TAG        0.3978
pareto_id               0
PHB_pFBA           0.3978
PHB_FBA            0.3978
                   ...   
mal_supplied    -0.238157
hxa_supplied          0.0
oct_supplied       -0.112
nh4_supplied          0.0
hco3_supplied      0.2696
Length: 74, dtype: object
------------------------------------------------------------
  TFA ACTIVE — 1 ΔG variables created for Pareto solution 1
    AACOAR_syn ΔG variable not found.
PHB_SYN_UB         0.3978
PHB_SYN_TAG        0.3978
pareto_id               1
PHB_pFBA           0.3978
PHB_FBA            0.3978
                   ...   
mal_supplied          0.0
hxa_supplied    -0.079939
oct_supplied          0.0
nh4_supplied          0.0
hco3_supplied         0.0
Length: 74, dtype: object
------------------------------------------------------------
  TFA ACTIVE — 1 ΔG variables cr

In [ ]:
from pytfa.optim.variables import DeltaG

dg_vars = mytfa.get_variables_of_type(DeltaG)

print("Number of ΔG variables:", len(dg_vars))


Number of ΔG variables: 1


In [ ]:
thermo_rxns = []

for v in dg_vars:
    thermo_rxns.append(v.reaction.id)

thermo_rxns = sorted(set(thermo_rxns))

print("Thermodynamically constrained reactions:")
print(thermo_rxns)
print("Total:", len(thermo_rxns))


Thermodynamically constrained reactions:
['NADTRHD']
Total: 1


In [ ]:
def has_dGr_variable(tmodel, rxn_id):
    return any(v.reaction.id == rxn_id for v in tmodel.get_variables_of_type(DeltaG))

for rxn_id in ["ACACT1r", "ACACCT", "AACOAR_syn", "HACD1_2", "PHBS_syn"]:
    print(rxn_id, "thermo constrained:", has_dGr_variable(tmodel, rxn_id))


NameError: name 'tmodel' is not defined

# *Compare results

In [ ]:
import pandas as pd
import numpy as np


CSV_FILE = PHB_TFA_DIR / f"05_FBA_TFA_PHB_detailed_results_0.3978.csv"

TOL = 1e-6

PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1",
]

df = pd.read_csv(CSV_FILE)

for idx, row in df.iterrows():

    print(f"\nPareto row: {row['pareto_id']} | Strategy: {row['optimization_strategy']}")
    print("-" * 60)

    for r in PHB_REACTIONS:

        pfba = row[f"{r}_pFBA"]
        tfa  = row[f"{r}_TFA"]

        if pd.isna(pfba) or pd.isna(tfa):
            relation = "missing values"

        elif np.isclose(pfba, tfa, atol=TOL):
            relation = "≈ equal"

        elif pfba > tfa:
            relation = "pFBA HIGHER than TFA"

        else:
            relation = "pFBA LOWER than TFA"

        print(f"{r:12s} → {relation}  (pFBA={pfba:.6f}, TFA={tfa:.6f})")


Pareto row: 0 | Strategy: SUP
------------------------------------------------------------
ACACT1r      → pFBA LOWER than TFA  (pFBA=0.128257, TFA=0.397800)
ACACCT       → ≈ equal  (pFBA=0.000000, TFA=0.000000)
AACOAR_syn   → ≈ equal  (pFBA=0.397800, TFA=0.397800)
HACD1_2      → ≈ equal  (pFBA=0.000000, TFA=0.000000)
HACD1        → ≈ equal  (pFBA=0.000000, TFA=0.000000)
HACD1i       → pFBA HIGHER than TFA  (pFBA=0.269543, TFA=0.000000)
KAT1         → ≈ equal  (pFBA=0.000000, TFA=0.000000)

Pareto row: 1 | Strategy: CON
------------------------------------------------------------
ACACT1r      → ≈ equal  (pFBA=0.397800, TFA=0.397800)
ACACCT       → ≈ equal  (pFBA=0.000000, TFA=0.000000)
AACOAR_syn   → ≈ equal  (pFBA=0.397800, TFA=0.397800)
HACD1_2      → ≈ equal  (pFBA=0.000000, TFA=0.000000)
HACD1        → ≈ equal  (pFBA=0.000000, TFA=0.000000)
HACD1i       → ≈ equal  (pFBA=0.000000, TFA=0.000000)
KAT1         → ≈ equal  (pFBA=0.000000, TFA=0.000000)

Pareto row: 2 | Strategy: SUP_and_

In [ ]:
import numpy as np
import pandas as pd
import cobra
import pytfa

from cobra.flux_analysis import pfba

from pytfa.io import (
    read_lexicon,
    annotate_from_lexicon,
    read_compartment_data,
    apply_compartment_data,
)
import logging

logging.getLogger("thermomodel_None").setLevel(logging.ERROR)
logging.getLogger("pytfa").setLevel(logging.ERROR)

# ============================================================
# USER PARAMETERS
# ============================================================
PHB_SYN_UB_VALUES =  [0.3978, 1000]
# ============================================================
# CONSTANTS
# ============================================================
PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1",
]

SUB_TO_RXN_BASE = {
    "ace": "EX_ac_e",
    "lac": "EX_lac__L_e",
    "ppa": "EX_ppa_e",
    "but": "EX_but_e",
    "ibt": "EX_ibt_e",
    "mal": "EX_mal__L_e",
    "hxa": "EX_hxa_e",
    "oct": "EX_octa_e",
    "nh4": "EX_nh4_e",
    "hco3": "EX_hco3_e",
}

EXCHANGE_RXNS = list(SUB_TO_RXN_BASE.values())

# ============================================================
# SOLVER SETTINGS
# ============================================================
def apply_solver_settings(model):
    model.solver = "glpk"
    model.solver.configuration.tolerances.feasibility = 1e-9
    model.solver.configuration.presolve = True


# ============================================================
# LOAD TFA METADATA (ONCE)
# ============================================================
lexicon = read_lexicon(str(lexicon_path))
comp_data = read_compartment_data(str(compartments_path))


# ============================================================
# MAIN LOOP
# ============================================================
for PHB_SYN_UB in PHB_SYN_UB_VALUES:

    print("\n==============================")
    print(f"PHB_SYN_UB_VALUE: {PHB_SYN_UB}")
    print("==============================")

    PHB_SYN_TAG = f"{PHB_SYN_UB:g}"

    INPUT_CSV = PHB_PARETO_DIR / f"03_pareto_best_all_strategies_{PHB_SYN_TAG}.csv"
    OUT_CSV = PHB_TFA_DIR / f"05_FBA_TFA_PFBA_detailed_results_all_strategies_{PHB_SYN_TAG}.csv"

    df = pd.read_csv(INPUT_CSV)
    results = []

    for pareto_id, row in df.iterrows():

        strategy = row["optimization_strategy"]

        if "SUP" in strategy:
            suffix = "_sup"

        elif strategy == "CON":
            suffix = "_con"

        elif strategy == "SUP_and_CON":
            suffix = "_sup"

        elif strategy == "PFBA":
            suffix = "_con_pfba"

        elif strategy == "SUP_and_PFBA":
            suffix = "_sup"

        else:
            print('Other optimization strategy')

        m = model_constrained.copy()

        # ---------------------------------
        # STORE INPUT VALUES USED
        # ---------------------------------
        input_values_used = {}

        for base_col, rxn_id in SUB_TO_RXN_BASE.items():

            col_name = f"{base_col}{suffix}"
            val_used = np.nan

            if col_name in row:
                val = row[col_name]
                if pd.notna(val) and float(val) != 0.0 and rxn_id in m.reactions:
                    rxn = m.reactions.get_by_id(rxn_id)
                    rxn.lower_bound = -abs(float(val))
                    rxn.upper_bound = 0.0
                    val_used = float(val)

            input_values_used[f"{base_col}_supplied"] = 0.0 if pd.isna(val_used) else val_used

        # ---------------------------------
        # OBJECTIVE & NO GROWTH
        # ---------------------------------
        phb_rxn = m.reactions.get_by_id("PHBS_syn")
        phb_rxn.lower_bound = 0.0
        phb_rxn.upper_bound = PHB_SYN_UB
        m.objective = phb_rxn

        biomass = m.reactions.get_by_id("BIOMASS__1")
        biomass.lower_bound = 0.0
        biomass.upper_bound = 0.0

        apply_solver_settings(m)

        # ---------------------------------
        # FBA
        # ---------------------------------
        fba_sol = m.optimize()

        # ---------------------------------
        # pFBA
        # ---------------------------------

        pfba_sol = pfba(m)

        # ---------------------------------
        # TFA
        # ---------------------------------
        # Ensure thermo_data metabolites keys are plain strings
        thermo_data['metabolites'] = {k: v for k, v in thermo_data['metabolites'].items()}

        mytfa = pytfa.ThermoModel(thermo_data, m)

        # Explicitly set compartments to avoid TypeError, mimicking cobra_model_mat setup
        mytfa.compartments = {
            'c': {
                'pH': 7.0,
                'ionicStr': 0.25,
                'c_min': 1e-6,
                'c_max': 0.02,
                'membranePot': 0.0,
            },
            'e': {
                'pH': 7.0,
                'ionicStr': 0.25,
                'c_min': 1e-6,
                'c_max': 0.02,
                'membranePot': 0.0,
            },
            'p': {
                'pH': 7.0,
                'ionicStr': 0.25,
                'c_min': 1e-6,
                'c_max': 0.02,
                'membranePot': 0.15,
            },
            'u': {
                'pH': 7.0,
                'ionicStr': 0.25,
                'c_min': 1e-6,
                'c_max': 0.02,
                'membranePot': 0.0,
            }
        }

        # These lines are crucial for TFA to work correctly
        annotate_from_lexicon(mytfa, lexicon)
        apply_compartment_data(mytfa, comp_data)

        mytfa.objective = "PHBS_syn"
        apply_solver_settings(mytfa)
        mytfa.prepare()
        mytfa.convert()

        # Check if DeltaG variables are created to confirm TFA is active
        from pytfa.optim.variables import DeltaG
        if DeltaG in mytfa.variables and len(mytfa.variables[DeltaG]) > 0:
            print(f"  TFA: DeltaG variables created. TFA is active for Pareto solution {pareto_id}.")
        else:
            print(f"  TFA: No DeltaG variables created. TFA is NOT active for Pareto solution {pareto_id}. (Falling back to FBA behavior)")

        tfa_sol = mytfa.optimize()

        cobra.io.write_sbml_model(
            mytfa,
            PHB_MODEL_TFA_DIR / f"model_TFA_PHB_{PHB_SYN_TAG}.xml"
        )

        # ---------------------------------
        # COLLECT RESULTS
        # ---------------------------------
        result = {
            "PHB_SYN_UB": PHB_SYN_UB,
            "PHB_SYN_TAG": PHB_SYN_TAG,
            "pareto_id": pareto_id,
            "PHB_pFBA": pfba_sol.fluxes.get("PHBS_syn", np.nan),
            "PHB_FBA": fba_sol.fluxes.get("PHBS_syn", np.nan),
            "PHB_TFA": tfa_sol.fluxes.get("PHBS_syn", np.nan),
            "Biomass_pFBA": pfba_sol.fluxes.get("BIOMASS__1", np.nan),
            "Biomass_FBA": fba_sol.fluxes.get("BIOMASS__1", np.nan),
            "Biomass_TFA": tfa_sol.fluxes.get("BIOMASS__1", np.nan),
            "status_pFBA": pfba_sol.status,
            "status_FBA": fba_sol.status,
            "status_TFA": tfa_sol.status,
            "optimization_strategy": strategy
        }

        for r in PHB_REACTIONS:
            result[f"{r}_pFBA"] = pfba_sol.fluxes.get(r, np.nan)
            result[f"{r}_FBA"] = fba_sol.fluxes.get(r, np.nan)
            result[f"{r}_TFA"] = tfa_sol.fluxes.get(r, np.nan)

        for base_col, rxn_id in SUB_TO_RXN_BASE.items():
            result[f"{base_col}_pFBA"] = pfba_sol.fluxes.get(rxn_id, np.nan)
            result[f"{base_col}_FBA"]  = fba_sol.fluxes.get(rxn_id, np.nan)
            result[f"{base_col}_TFA"]  = tfa_sol.fluxes.get(rxn_id, np.nan)

        result.update(input_values_used)
        print(pd.Series(result))
        print("-" * 60)
        results.append(result)

    pd.DataFrame(results).to_csv(OUT_CSV, index=False)
    print(f"\nDONE — Results saved to:\n{OUT_CSV}")


# *** Additional code ***



# Check TFA model

In [ ]:
assert any(c.name.startswith("G_") for c in mytfa.constraints)
assert any(v.name.startswith("DGo_") for v in mytfa.variables)
assert any(v.type == "binary" for v in mytfa.variables)
print("✅ TFBA is active")

✅ TFBA is active


In [ ]:
for v in mytfa.variables.values():
  print(v)

  #0.0 <= DM_4CRSOL <= 1000.0


0.0 <= QULNS <= 1000.0
0.0 <= QULNS_reverse_66da1 <= 0.0
0.0 <= ORNDC <= 1000.0
0.0 <= ORNDC_reverse_63596 <= 0.0
0.0 <= MSBENZMT <= 1000.0
0.0 <= MSBENZMT_reverse_a902a <= 0.0
0.0 <= DESAT18a <= 1000.0
0.0 <= DESAT18a_reverse_fd859 <= 1000.0
0.0 <= FUM <= 1000.0
0.0 <= FUM_reverse_d3642 <= 1000.0
0.0 <= PHYFXOR <= 1000.0
0.0 <= PHYFXOR_reverse_84960 <= 0.0
0.0 <= VPAMTr <= 1000.0
0.0 <= VPAMTr_reverse_872bd <= 0.0
0.0 <= GLYCL <= 1000.0
0.0 <= GLYCL_reverse_e418f <= 0.0
0.0 <= NDPK7 <= 1000.0
0.0 <= NDPK7_reverse_9dc79 <= 1000.0
0.0 <= GTPCI <= 1000.0
0.0 <= GTPCI_reverse_1ee86 <= 0.0
0.0 <= MTHFC <= 1000.0
0.0 <= MTHFC_reverse_f6fcc <= 1000.0
0.0 <= GCATENEC <= 1000.0
0.0 <= GCATENEC_reverse_ae4a5 <= 1000.0
0.0 <= ORNTA <= 1000.0
0.0 <= ORNTA_reverse_5adff <= 1000.0
0.0 <= UPPDC1 <= 1000.0
0.0 <= UPPDC1_reverse_cb592 <= 0.0
0.0 <= GARFT <= 1000.0
0.0 <= GARFT_reverse_7ecb6 <= 0.0
0.0 <= H4THDPR <= 1000.0
0.0 <= H4THDPR_reverse_617be <= 0.0
0.0 <= UDPG4E <= 1000.0
0.0 <= UDPG4E_revers

In [ ]:
[v.name for v in mytfa.variables if v.name.startswith("LC_")]



['LC_gam6p_c',
 'LC_cgly_c',
 'LC_achms_c',
 'LC_pcox_u',
 'LC_octe9ACP_c',
 'LC_fdp_c',
 'LC_5caiz_c',
 'LC_pep_c',
 'LC_coa_c',
 'LC_hgbam_c',
 'LC_nac_c',
 'LC_r3mmal_c',
 'LC_leu__L_c',
 'LC_ahcys_c',
 'LC_ru5p__D_c',
 'LC_14dhncoa_c',
 'LC_h2o_cx_c',
 'LC_h2s_c',
 'LC_orot_c',
 'LC_g1p_c',
 'LC_3hdecACP_c',
 'LC_na1_c',
 'LC_2pglyc_c',
 'LC_ala_B_c',
 'LC_glyc3p_c',
 'LC_pqh2_um_p',
 'LC_26dap_LL_c',
 'LC_ddcaACP_c',
 'LC_2pglyc_cx_c',
 'LC_trdox_c',
 'LC_3oddecACP_c',
 'LC_glu__D_c',
 'LC_h_cx_c',
 'LC_dutp_c',
 'LC_paps_c',
 'LC_mg2_c',
 'LC_hdeACP_c',
 'LC_o2_c',
 'LC_anth_c',
 'LC_dhpt_c',
 'LC_thex2eACP_c',
 'LC_fmn_c',
 'LC_pppi_c',
 'LC_udp_c',
 'LC_cit_c',
 'LC_acACP_c',
 'LC_uama_c',
 'LC_eig3p_c',
 'LC_3c4mop_c',
 'LC_lys__L_c',
 'LC_13dpg_c',
 'LC_gmp_c',
 'LC_utp_c',
 'LC_trdrd_c',
 'LC_so4_c',
 'LC_5apru_c',
 'LC_hmppp9_c',
 'LC_pydx5p_c',
 'LC_2dhp_c',
 'LC_ade_c',
 'LC_asp__L_c',
 'LC_2sephchc_c',
 'LC_acg5p_c',
 'LC_gdp_c',
 'LC_hgbyr_c',
 'LC_palmACP_c',
 'LC_pser

# Find reactions for building lexicon for iDT1294

In [ ]:
# Add PHB_REACTIONS

# Acetyl-CoA C-acetyltransferase
#Acetyl-CoA:acetoacetyl-CoA transferase
#Acetoacetyl CoA reductase
#3 hydroxyacyl CoA dehydrogenase  acetoacetyl CoA
#3-ketoacyl-CoA thiolase

PHB_REACTIONS = [
    "ACACT1r",
    "ACACCT",
    "AACOAR_syn",
    "HACD1_2",
    "HACD1",
    "HACD1i",
    "KAT1"
]

phb_mets = set()

for rxn_id in PHB_REACTIONS:
    rxn = cobra_model_mat.reactions.get_by_id(rxn_id)
    for met in rxn.metabolites:
        phb_mets.add(met)

for met in sorted(phb_mets, key=lambda m: m.id):
    print(f"{met.id}\t{met.name}\t{met.formula}\t{met.compartment}")




3hbcoa__R_c	(R)-3-Hydroxybutyryl-CoA	C25H39N7O18P3S	c
3hbcoa_c	(S)-3-Hydroxybutanoyl-CoA	C25H38N7O18P3S	c
3hbycoa_c	S  3 Hydroxybutyryl CoA C25H38N7O18P3S	C25H38N7O18P3S	c
aacoa_c	Acetoacetyl-CoA	C25H37N7O18P3S	c
ac_c	Acetate	C2H3O2	c
acac_c	Acetoacetate	C4H5O3	c
accoa_c	Acetyl-CoA	C23H35N7O17P3S	c
coa_c	Coenzyme A	C21H33N7O16P3S	c
h_c	H+	H	c
nad_c	Nicotinamide adenine dinucleotide	C21H26N7O14P2	c
nadh_c	Nicotinamide adenine dinucleotide - reduced	C21H27N7O14P2	c
nadp_c	Nicotinamide adenine dinucleotide phosphate	C21H26N7O17P3	c
nadph_c	Nicotinamide adenine dinucleotide phosphate - reduced	C21H27N7O17P3	c


In [ ]:
from pytfa.io import annotate_from_lexicon

lexicon = pd.read_csv("phb_lexicon.csv")

annotate_from_lexicon(
    mytfa,
    lexicon,
    inplace=True
)


## Check TFA R. palustris

In [ ]:
# Convert DataFrame → dict lexicon
lexicon_dict = (
    lexicon['seed_id']
    .dropna()
    .astype(str)
    .to_dict()
)

print(type(lexicon_dict))
print(len(lexicon_dict))
print(list(lexicon_dict.items())[:5])


<class 'dict'>
8
[('glc__D_c', 'cpd00027'), ('pyr_c', 'cpd00020'), ('accoa_c', 'cpd00022'), ('atp_c', 'cpd00002'), ('adp_c', 'cpd00008')]


In [ ]:
import pandas as pd

rows = []
for m in cobra_model_mat.metabolites:
    rows.append({
        "met_id": m.id,
        "name": m.name,
        "formula": m.formula,
        "charge": m.charge,
        "compartment": m.compartment
    })

df = pd.DataFrame(rows)
df.to_csv("metabolites_for_mapping.csv", index=False)


# Assing deltaG0_prime to important reactions

In [ ]:
from pytfa.optim.variables import DeltaG

PHB_REACTION_THERMO = {
    "ACACT1r": {      # 2 acetyl-CoA ⇌ acetoacetyl-CoA + CoA
        "delta_g0": +18.0,
        "uncertainty": 5.0
    },

    "AACOAR_syn": {   # acetoacetyl-CoA + NADPH → 3HB-CoA + NADP+
        "delta_g0": -25.0,
        "uncertainty": 5.0
    },

    "PHBS_syn": {     # polymerization (CoA-releasing)
        "delta_g0": -35.0,
        "uncertainty": 7.0
    },

    "ACACCT": {       # CoA transfer / activation
        "delta_g0": -10.0,
        "uncertainty": 5.0
    }
}


for rxn_id in PHB_REACTION_THERMO:
    rxn = tmodel.reactions.get_by_id(rxn_id)
    dg_vars = [v for v in tmodel.variables.values() if isinstance(v, DeltaG) and v.reaction == rxn]
    print(rxn_id, "ΔG vars:", dg_vars)

ACACT1r ΔG vars: []
AACOAR_syn ΔG vars: []
PHBS_syn ΔG vars: []
ACACCT ΔG vars: []


In [ ]:
thermo_data2['cues'].update({
    "AACOAR_syn": {"delta_g0": -25.0, "uncertainty": 5.0},
    "PHBS_syn":    {"delta_g0": -20.0, "uncertainty": 5.0},
    "ACACT1r":     {"delta_g0": -10.0, "uncertainty": 5.0},
    "ACACCT":      {"delta_g0": -15.0, "uncertainty": 5.0},
})


In [ ]:
# Reactions of interest
phb_reactions = ["AACOAR_syn", "ACACT1r"]

thermo_mets = thermo_data["metabolites"]  # ThermoDB compounds dict

def normalize(s):
    return s.lower().replace("-", "").replace("_", "").replace(" ", "")

print("\n=== PHB Reactions: On-the-fly ThermoDB mapping ===")

for rxn_id in phb_reactions:
    if rxn_id not in model.reactions:
        print(f"\n Reaction {rxn_id} not in model")
        continue

    rxn = model.reactions.get_by_id(rxn_id)
    print(f"\nReaction: {rxn_id}")
    print(rxn.build_reaction_string())
    print("-" * 70)

    for met in rxn.metabolites:
        print(f"\nMetabolite: {met.id}")
        print(f"  Name:        {met.name}")
        print(f"  Formula:     {met.formula}")
        print(f"  Compartment: {met.compartment}")

        formula_hits = []
        name_hits = []

        for seed_id, tmeta in thermo_mets.items():
            # Formula match
            if met.formula and tmeta.get("formula") == met.formula:
                formula_hits.append((seed_id, tmeta.get("name")))

            # Name match (loose)
            if met.name and normalize(met.name) in normalize(tmeta.get("name", "")):
                name_hits.append((seed_id, tmeta.get("name")))

        if formula_hits:
            print("  🔬 ThermoDB formula matches:")
            for sid, name in formula_hits[:5]:
                print(f"     - seed_id: {sid} | {name}")
        else:
            print("   No ThermoDB formula matches")

        if name_hits:
            print("  🧬 ThermoDB name matches:")
            for sid, name in name_hits[:5]:
                print(f"     - seed_id: {sid} | {name}")
        else:
            print("  ❌ No ThermoDB name matches")



=== PHB Reactions: On-the-fly ThermoDB mapping ===

Reaction: AACOAR_syn
aacoa[c] + h[c] + nadph[c] --> 3hbcoa__R[c] + nadp[c]
----------------------------------------------------------------------

Metabolite: h[c]
  Name:        H+
  Formula:     H
  Compartment: c
  🔬 ThermoDB formula matches:
     - seed_id: h_cx[c] | None
     - seed_id: h[c] | None
  ❌ No ThermoDB name matches

Metabolite: nadph[c]
  Name:        Nicotinamide adenine dinucleotide phosphate - reduced
  Formula:     C21H27N7O17P3
  Compartment: c
  🔬 ThermoDB formula matches:
     - seed_id: nadph[c] | None
  ❌ No ThermoDB name matches

Metabolite: aacoa[c]
  Name:        Acetoacetyl-CoA
  Formula:     C25H37N7O18P3S
  Compartment: c
  🔬 ThermoDB formula matches:
     - seed_id: aacoa[c] | None
  ❌ No ThermoDB name matches

Metabolite: nadp[c]
  Name:        Nicotinamide adenine dinucleotide phosphate
  Formula:     C21H26N7O17P3
  Compartment: c
  🔬 ThermoDB formula matches:
     - seed_id: nadp[c] | None
  ❌ No 

In [ ]:
# List of reactions of interest
phb_reactions = ["AACOAR_syn", "ACACT1r"] #"PHBS_syn","ACACCT"

print("=== PHB Reactions Metabolites ===")
for rxn_id in phb_reactions:
    if rxn_id not in model.reactions:
        print(f"Reaction {rxn_id} not in model")
        continue

    rxn = model.reactions.get_by_id(rxn_id)
    print(f"\nReaction: {rxn_id} | Equation: {rxn.build_reaction_string()}")
    print(f"{'Metabolite ID':<20} | {'Formula':<15} | {'Compartment'}")
    print("-"*55)

    for met, coeff in rxn.metabolites.items():
        # check if formula exists
        formula = met.formula if met.formula is not None else "None"
        print(f"{met.id:<20} | {formula:<15} | {met.compartment}")


=== PHB Reactions Metabolites ===

Reaction: AACOAR_syn | Equation: aacoa[c] + h[c] + nadph[c] --> 3hbcoa__R[c] + nadp[c]
Metabolite ID        | Formula         | Compartment
-------------------------------------------------------
h[c]                 | H               | c
nadph[c]             | C21H27N7O17P3   | c
aacoa[c]             | C25H37N7O18P3S  | c
nadp[c]              | C21H26N7O17P3   | c
3hbcoa__R[c]         | C25H39N7O18P3S  | c

Reaction: ACACT1r | Equation: 2.0 accoa[c] --> aacoa[c] + coa[c]
Metabolite ID        | Formula         | Compartment
-------------------------------------------------------
accoa[c]             | C23H35N7O17P3S  | c
coa[c]               | C21H33N7O16P3S  | c
aacoa[c]             | C25H37N7O18P3S  | c


In [ ]:
from pytfa.optim.variables import DeltaG

for rxn_id in PHB_REACTION_THERMO:
    rxn = mytfa.reactions.get_by_id(rxn_id)

    dg_vars = [
        v for v in mytfa.variables.values()
        if isinstance(v, DeltaG) and v.reaction == rxn
    ]

    print(rxn_id, "ΔG vars:", dg_vars)


ACACT1r ΔG vars: []
AACOAR_syn ΔG vars: []
PHBS_syn ΔG vars: []
ACACCT ΔG vars: []


# Check up thermodata

In [ ]:
# CHECK METABOLITES FORMULAS (OR NAME)
names = [entry['formula'] for entry in thermo_data['metabolites'].values()]
thermo_formulas = [str(f) for f in names if f is not None]
print(thermo_formulas)

['C6H12O9P', 'C37H56N8O9', 'CH3Br', 'C19H15N3O7', 'C21H26N2O6Cl2', 'NA', 'C18H16N6O8S3', 'C5H10O8P', 'C30H50O4', 'C22H29N7O5', 'C15H28N2', 'C27H46O', 'C22H16N4O', 'C7H17N4O', 'C36H61N7O17P3S', 'C19H21N7O6', 'C30H51N7O7', 'C23H31NO7', 'C9H14N4O4', 'C10H14CaO6', 'C15H26O2', 'C3H4N2', 'H2Cu2O', 'C6H7O3RS', 'C8H11NO', 'C14H14O4', 'C16H20N2O2', 'C20H28O', 'C28H50O2', 'C16H17NO2', 'C40H60', 'C7H9N2', 'C11H18N2O2S', 'C20H30N6O12S2', 'C16H21N4O9', 'C15H16O2', 'C60H78OSn2', 'C12H19N2O', 'C9H11O11PR3', 'C18H26O4', 'C8H16N2O4Se2', 'C13H20O', 'C9H16N4OS', 'C6H14O6', 'C7H10N2O3', 'C10H12O2', 'C20H32', 'NA', 'NA', 'C27H34O5', 'C9H9O5', 'C15H20O3', 'C35H32N4O5Mg', 'C35H44O16', 'C12H6O4', 'CNR', 'C20H24O6', 'C8H18O4S4', 'C8H10O', 'NA', 'C23H18O8', 'C11H16O16P2R2', 'C22H24N3OCl', 'C4H8O2', 'C19H28O3', 'C40H71N3O15P2', 'C8H8NO4', 'C25H39N7O18P3S', 'C23H26O8', 'C30H50O', 'C16H16O11', 'C17H22N3', 'C11H17O2', 'NA', 'C19H11O2', 'C18H26NO6', 'C24H47NO10S', 'C13H16N2O3', 'C19H25N2O', 'C20H25N2OS', 'C34H63N2O6

In [ ]:
# Look for metabolites with specific composition

target_elements = ["C21", "N7", "P2"]

for entry in thermo_formulas:
    if all(elem in entry for elem in target_elements):
        print(entry)


C21H27N7O14P2
C21H44N7O18P2
C21H29N7O15P2
C21H43N7O18P2
C21H26N7O14P2
C21H33N7O13P2S
C21H44N7O17P2
C21H44N7O17P2
C21H44N7O18P2


In [ ]:
# LOOCKUP INFORMATION OF SPECIFIC METABOLITES USING THEIR FORMULA

metabolites_to_check = [
    "C21H27N7O14P2",
    "C21H29N7O15P2",
    "C21H26N7O14P2",
]

for met in metabolites_to_check:
    found = False
    for met_id, entry in thermo_data['metabolites'].items():
        name_lower = entry.get('formula', '').lower()
        # Exact match
        if met.lower() == name_lower:
            print(f"Metabolite ID: {met_id}")
            for key, value in entry.items():
                print(f"  {key}: {value}")
            print("-"*50)
            found = True
            break  # stop after the first exact match
    if not found:
        print(f"{met}  NOT found in thermo_data")
        print("-"*50)

Metabolite ID: cpd00004
  pKa: [1.8, 2.56, 12.69, 13.31, 14.3]
  deltaGf_err: 4.2679
  mass_std: 663.0
  struct_cues: {'RWWNW': 2, 'mid_phos': 1, 'WNH2': 2, 'amide': 1, 'RWCHWW': 8, 'RWOW': 2, 'RWCHdblW': 5, 'WPO4nW': 1, 'TWWCdblW': 2, 'Origin': 1, 'WketoneW': 1, 'WCH2W': 2, 'RWdblNW': 3, 'RWCH2W': 1, 'OCCC': 1, 'HeteroAromatic': 2, 'PrimOH': 4, 'RWCdblWW': 2}
  id: cpd00004
  nH_std: 27
  name: NADH
  formula: C21H27N7O14P2
  deltaGf_std: -524.32
  error: Nil
  charge_std: -2
  other_names: ['NADH', 'DPNH', 'Nicotinamide adenine dinucleotide - reduced', 'Nicotinamideadeninedinucleotide-reduced', 'nadh']
--------------------------------------------------
Metabolite ID: cpd02951
  pKa: [1.8, 2.56, 12.69, 13.31, 14.3]
  deltaGf_err: 4.238
  mass_std: 681.0
  struct_cues: {'RWWNW': 2, 'mid_phos': 1, 'WNH2': 2, 'amide': 1, 'RWCHWW': 9, 'RWOW': 2, 'RWCHdblW': 3, 'WPO4nW': 1, 'TWWCdblW': 2, 'Origin': 1, 'WketoneW': 1, 'WCH2W': 2, 'RWdblNW': 3, 'RWCH2W': 2, 'OCCC': 1, 'HeteroAromatic': 2, 'Pr

# Load thermodynamics database

In [ ]:
# Download thermo database
!wget https://raw.githubusercontent.com/EPFL-LCSB/pytfa/master/data/thermo_data.thermodb -O thermo_data.thermodb

# Load thermodynamics
thermo_data = load_thermoDB("thermo_data.thermodb")

# Print thermo_data keys

print(f"Thermo Data loaded successfully:{thermo_data.keys()}")

--2026-01-04 16:01:23--  https://raw.githubusercontent.com/EPFL-LCSB/pytfa/master/data/thermo_data.thermodb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2184378 (2.1M) [application/octet-stream]
Saving to: ‘thermo_data.thermodb’

thermo_data.thermod 100%[===================>]   2.08M  --.-KB/s    in 0.07s   

2026-01-04 16:01:24 (31.0 MB/s) - ‘thermo_data.thermodb’ saved [2184378/2184378]

Thermo Data loaded successfully:dict_keys(['name', 'units', 'metabolites', 'cues'])


In [ ]:
# Adjust model compartments as required by TFA

model.compartments = {
    'c': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },
    'e': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },

    'u': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },
    'p': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    }
}

# Step 1: map metabolites to standard compartments
for met in model.metabolites:
    if met.compartment in ["c", "cytosol"]:
        met.compartment = "c"
    elif met.compartment in ["e", "extracellular"]:
        met.compartment = "e"

# Step 2: collect all compartments used by metabolites
used_compartments = set([met.compartment for met in model.metabolites])

# Step 3: ensure all compartments exist and are dicts
for c in used_compartments:
    val = model.compartments.get(c)
    # overwrite any string or missing compartment with dict
    model.compartments[c] = {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02
    }

print("Final compartments:", model.compartments)


Final compartments: {'c': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'u': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'p': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'e': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}}


In [ ]:
rxn = model.reactions.get_by_id('AACOAR_syn')
rxn.lower_bound = -1000
rxn.upper_bound = 1000

for rxn in model.reactions:
    rxn.lower_bound = max(rxn.lower_bound, -1000)
    rxn.upper_bound = min(rxn.upper_bound, 1000)


tmodel = ThermoModel(
    thermo_data,
    model
)

for comp in tmodel.compartments:
    tmodel.compartments[comp] = {
        "pH": 7.0,
        "ionicStr": 0.25
    }

tmodel.solver = "glpk" #'optlang-cplex'


# Set physiological concentration ranges
tmodel.metabolites.get_by_id("nadph[c]").concentration = (1e-6, 1e-3)
tmodel.metabolites.get_by_id("nadp[c]").concentration  = (1e-6, 1e-3)
tmodel.metabolites.get_by_id("h[c]").concentration     = (1e-8, 1e-6)
tmodel.metabolites.get_by_id("coa[c]").concentration   = (1e-6, 1e-3)
#for met in tmodel.metabolites:
#    met.concentration = (1e-9, 1e-2)


tmodel.prepare()
tmodel.convert()



In [ ]:
# TFA OPTIMIZATION

sol = tmodel.optimize()



DEBUG:thermomodel_Model Exported from COBRA Toolbox:'slim_optimize' ((), {}) 441.55 sec


In [ ]:
# Explain why thi is necessary
rxn = tmodel.reactions.get_by_id('AACOAR_syn')
rxn.thermo['deltaG0_prime'] = (-25.0, 2.0)
rxn.thermo

{'isTrans': False,
 'computed': False,
 'deltaGR': 1000.0,
 'deltaGRerr': 1000.0,
 'deltaG0_prime': (-25.0, 2.0)}

In [ ]:
rxn = tmodel.reactions.get_by_id('AACOAR_syn')
rxn.thermo

{'isTrans': False,
 'computed': False,
 'deltaGR': 1000.0,
 'deltaGRerr': 1000.0,
 'deltaG0_prime': (-25.0, 2.0)}

# Formula normalization and matching

In [ ]:
gem_formulas = {
    m.id: m.formula
    for m in model.metabolites
    if m.formula is not None
}
gem_formulas

In [ ]:
# Normalize Formula using Hill notation
import re
from collections import defaultdict

def normalize_formula(formula):
    """
    Normalize chemical formula to Hill-like notation.
    Example: 'H12C6O6P1' -> 'C6H12O6P'
    """
    if formula is None:
        return None

    # Parse elements and counts
    tokens = re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    if not tokens:
        return None

    elements = defaultdict(int)
    for el, count in tokens:
        elements[el] += int(count) if count else 1

    # Hill system: C, then H, then others alphabetically
    ordered = []
    if 'C' in elements:
        ordered.append(('C', elements.pop('C')))
    if 'H' in elements:
        ordered.append(('H', elements.pop('H')))
    for el in sorted(elements):
        ordered.append((el, elements[el]))

    return ''.join(f"{el}{cnt if cnt > 1 else ''}" for el, cnt in ordered)


In [ ]:
# Find
thermo_norm = set(
    normalize_formula(f) for f in thermo_formulas
    if normalize_formula(f) is not None
)

gem_norm = {
    mid: normalize_formula(f)
    for mid, f in gem_formulas.items()
    if normalize_formula(f) is not None
}


In [ ]:
matched = {
    mid: f for mid, f in gem_norm.items()
    if f in thermo_norm
}


In [ ]:
len(matched)

1412

In [ ]:
unmatched = {
    mid: f for mid, f in gem_norm.items()
    if f not in thermo_norm
}


In [ ]:
len(unmatched)

706

In [ ]:
for mid, f in list(matched.items())[:20]:
    print(mid, f)


gam6p[c] C6H13NO8P
cgly[c] C5H10N2O3S
achms[c] C6H11NO4
pcox_u R
octe9ACP[c] C18H33ORS
pep[c] C3H2O6P
hgbam[c] C45H58N6O12
nac[c] C6H4NO2
r3mmal[c] C5H6O5
leu__L[c] C6H13NO2
ahcys[c] C14H20N6O5S
h2o_cx[c] H2O
orot[c] C5H3N2O4
na1[c] Na
ala_B[c] C3H7NO2
pqh2_um[p] C53H82O2
26dap_LL[c] C7H14N2O4
ddcaACP[c] C12H23ORS
trdox[c] X
glu__D[c] C5H8NO4


In [ ]:
# CHANGE FORMULA MANUALLY

met = model.metabolites.get_by_id('coa[c]')
met.formula = 'C21H36N7O16P3S'
# keep the charge as is


# Match metabolites by name

In [ ]:
'Acetyl-CoA' in thermo_formulas

True

In [ ]:
metabolites_to_check = [
    "nadph",
    "nadh",
    "acetyl-coa",
    "nad+",
    "nadp+",
    "3-hydroxybutyryl-coa",
    "acetoacetyl-coa",
    "coa"
]

for met in metabolites_to_check:
    found = False
    for met_id, entry in thermo_data['metabolites'].items():
        name_lower = entry.get('name', '').lower()
        # Exact match
        if met.lower() == name_lower:
            print(f"Metabolite ID: {met_id}")
            for key, value in entry.items():
                print(f"  {key}: {value}")
            print("-"*50)
            found = True
            break  # stop after the first exact match
    if not found:
        print(f"{met} ❌ NOT found in thermo_data")
        print("-"*50)

Metabolite ID: cpd00005
  pKa: []
  deltaGf_err: 4.2579
  mass_std: 742.0
  struct_cues: {'RWWNW': 2, 'mid_phos': 1, 'prim_phos': 1, 'amide': 1, 'RWCHWW': 8, 'TWWCdblW': 2, 'RWCHdblW': 5, 'WPO4nW': 1, 'WNH2': 2, 'RWOW': 2, 'Origin': 1, 'WketoneW': 1, 'WCH2W': 2, 'RWdblNW': 3, 'RWCH2W': 1, 'OCCC': 1, 'HeteroAromatic': 2, 'PrimOH': 3, 'RWCdblWW': 2}
  id: cpd00005
  nH_std: 27
  name: NADPH
  formula: C21H27N7O17P3
  deltaGf_std: -736.82
  error: Nil
  charge_std: -3
  other_names: ['NADPH', 'TPNH', 'Nicotinamide adenine dinucleotide phosphate - reduced', 'Nicotinamideadeninedinucleotidephosphate-reduced', 'nadph']
--------------------------------------------------
Metabolite ID: cpd00004
  pKa: [1.8, 2.56, 12.69, 13.31, 14.3]
  deltaGf_err: 4.2679
  mass_std: 663.0
  struct_cues: {'RWWNW': 2, 'mid_phos': 1, 'WNH2': 2, 'amide': 1, 'RWCHWW': 8, 'RWOW': 2, 'RWCHdblW': 5, 'WPO4nW': 1, 'TWWCdblW': 2, 'Origin': 1, 'WketoneW': 1, 'WCH2W': 2, 'RWdblNW': 3, 'RWCH2W': 1, 'OCCC': 1, 'HeteroAromati

# Alternative version fro adding compartments

In [ ]:
# Adjust model compartments as required by TFA

model.compartments = {
    'c': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },
    'e': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },

    'u': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    },
    'p': {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02,
    }
}

# Step 1: map metabolites to standard compartments
for met in model.metabolites:
    if met.compartment in ["c", "cytosol"]:
        met.compartment = "c"
    elif met.compartment in ["e", "extracellular"]:
        met.compartment = "e"

# Step 2: collect all compartments used by metabolites
used_compartments = set([met.compartment for met in model.metabolites])

# Step 3: ensure all compartments exist and are dicts
for c in used_compartments:
    val = model.compartments.get(c)
    # overwrite any string or missing compartment with dict
    model.compartments[c] = {
        'pH': 7.0,
        'ionicStr': 0.25,
        'c_min': 1e-6,
        'c_max': 0.02
    }

print("Final compartments:", model.compartments)


Final compartments: {'c': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'u': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'p': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}, 'e': {'pH': 7.0, 'ionicStr': 0.25, 'c_min': 1e-06, 'c_max': 0.02}}


# Match reactions

In [ ]:
# CHECK PRECURSOR REACTIONS
precursor_rxns = [
    "AACOAR_syn",
    "PHBS_syn",
    "ACACT1r",
    "ACACCT",
]

for rxn_id in precursor_rxns:
    if any(rxn_id in v.name for v in dg_vars):
        print(f"{rxn_id}: thermo-constrained ✅")
    else:
        print(f"{rxn_id}: NOT thermo-constrained ❌")


AACOAR_syn: NOT thermo-constrained ❌
PHBS_syn: NOT thermo-constrained ❌
ACACT1r: NOT thermo-constrained ❌
ACACCT: NOT thermo-constrained ❌


In [ ]:
# Assuming your model is called 'model' and it's a COBRApy model

# Get the metabolite
met = model.metabolites.get_by_id("3hbcoa__R[c]")

# Find reactions where this metabolite is produced (stoichiometry > 0)
producing_reactions = [rxn for rxn in met.reactions if rxn.get_coefficient(met) > 0]

# Print them
for rxn in producing_reactions:
    print(rxn.id, rxn.reaction)


AACOAR_syn aacoa[c] + h[c] + nadph[c] --> 3hbcoa__R[c] + nadp[c]


In [ ]:


# Re-run prepare and convert
tmodel.prepare()
tmodel.convert()


2025-12-30 06:15:19,195 - thermomodel_Model Exported from COBRA Toolbox - INFO - # Model preparation starting...
INFO:thermomodel_Model Exported from COBRA Toolbox:# Model preparation starting...
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite gam6p_c (D-Glucosamine 6-phosphate) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite cgly_c (Cys Gly C5H10N2O3S) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite achms_c (O Acetyl L homoserine C6H11NO4) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite pcox_u (Plastocyanin(Cu2+)) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite octe9ACP_c (Cis-octadec-9-enoyl-[acyl-carrier protein] ((9Z)-n-C18:1)) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite fdp_c (D-Fructose 1,6-bisphosphate) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite 5caiz_c (5-phosphoribosyl-5-carbo

DEBUG:thermomodel_Model Exported from COBRA Toolbox:ENO : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PRFGS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PC8XM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:3OAS60 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GMPS2 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:RBFK : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NTRIRfx : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GTHPi : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:ARGSS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:SPTc : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:H2Otpp : thermo constraint NOT created
DEBUG:ther

DEBUG:thermomodel_Model Exported from COBRA Toolbox:GLUDGE_OLE_PALM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DGDGS_OLE_PALM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:H2Otcx : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GLNTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:TYRTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:METTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:SERTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GLYTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PROTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:CYSTRS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:ARGTRS : thermo const

DEBUG:thermomodel_Model Exported from COBRA Toolbox:PYDXO : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PYK2 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PYK3 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PYK4 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PYK5 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:R05224_1 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:RBCh : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:RBFSb : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:RBPC : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:SDPTA : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:SERD_L : thermo constraint NOT created
DEBUG:thermo

DEBUG:thermomodel_Model Exported from COBRA Toolbox:MOCOS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MOGDS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MPTS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MPTSS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MSAR : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MSO3abcpp : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:MTHFR2 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NADDP : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NADH10 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NADH16pp : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NADH17pp : thermo constraint NOT created


DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_ethso3_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_etoh_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_feenter_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_for_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_fru_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_g3pc_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_g3pi_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_g3ps_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_galct__D_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EX_glc__D_e : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox

Streaming output truncated to the last 5000 lines.
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added constraint: UF_MDDCP1ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added constraint: UR_MDDCP1ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating only use constraints for reactionMDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: FU_MDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: BU_MDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added constraint: SU_MDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added constraint: UF_MDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added constraint: UR_MDDCP4ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating only use constraints for reactionMDDCP5ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: FU_MDDCP5ex
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: BU_MDDCP5ex
DEBUG:the

In [ ]:
from pytfa.optim.variables import DeltaG

print("ΔG present:", DeltaG in tmodel.variables)


ΔG present: False


In [ ]:
rxn = tmodel.reactions.get_by_id("AACOAR_syn")

# Manually define standard Gibbs energy (kJ/mol)
tmodel.add_reaction_thermo(
    rxn,
    delta_g0=-25.0,    # literature estimate
    uncertainty=5.0
)


AttributeError: 'ThermoModel' object has no attribute 'add_reaction_thermo'

In [ ]:
print("ΔG variables present:", DeltaG in tmodel.variables)

if DeltaG not in tmodel.variables:
    raise RuntimeError(
        "❌ No DeltaG variables created → thermo_data not matched"
    )


ΔG variables present: False


RuntimeError: ❌ No DeltaG variables created → thermo_data not matched

In [ ]:
tmodel.metabolites.get_by_id("nadph[c]").concentration = (1e-6, 1e-3)
tmodel.metabolites.get_by_id("nadp[c]").concentration  = (1e-6, 1e-3)
tmodel.metabolites.get_by_id("h[c]").concentration     = (1e-8, 1e-6)
tmodel.metabolites.get_by_id("coa[c]").concentration   = (1e-6, 1e-3)


KeyError: 'nadph_c'

In [ ]:
rxn_id = "AACOAR_syn"
rxn = tmodel.reactions.get_by_id(rxn_id)

dg = tmodel.variables[DeltaG][rxn]

print(f"{rxn_id} ΔG bounds (kJ/mol):")
print("  LB:", dg.lb)
print("  UB:", dg.ub)


In [ ]:
# Solve
# FBA max PHB
fba = model.optimize()

# TFA max PHB
tfa = tmodel.optimize()
print("Thermo-FBA objective:", tfa.objective_value)

DEBUG:thermomodel_Model Exported from COBRA Toolbox:'slim_optimize' ((), {}) 91.40 sec


Thermo-FBA objective: 0.18


In [ ]:
# Reaction ID in your GEM
rxn = tmodel.reactions.get_by_id("AACOAR_syn")


# Add standard Gibbs energy (kJ/mol)
tmodel.add_reaction_thermo(
    rxn,
    delta_g0=-25.0,   # literature-based estimate
    uncertainty=5.0
)

In [ ]:
# --------------------------------------
# 2. PREPARE + CONVERT (safe even if partial)
# --------------------------------------
tmodel.convert()

2025-12-30 02:59:25,442 - thermomodel_Model Exported from COBRA Toolbox - INFO - # Model preparation starting...
INFO:thermomodel_Model Exported from COBRA Toolbox:# Model preparation starting...
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite gam6p[c] (D-Glucosamine 6-phosphate) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite cgly[c] (Cys Gly C5H10N2O3S) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite achms[c] (O Acetyl L homoserine C6H11NO4) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite pcox_u (Plastocyanin_Cu2+) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite octe9ACP[c] (Cis-octadec-9-enoyl-_acyl-carrier protein __9Z-n-C18:1) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite fdp[c] (D-Fructose 1,6-bisphosphate) has no seed_id
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Metabolite 5caiz[c] (5-phosphoribosyl-5-car

DEBUG:thermomodel_Model Exported from COBRA Toolbox:NTPP8 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PC11M : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:ARGabcpp : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PMDPHT : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DAPE : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:LYCOPC : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:RB15BPtcx : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GLCBRAN3 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DB4PS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DXPRIi : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GTPCII : thermo constraint NOT creat

DEBUG:thermomodel_Model Exported from COBRA Toolbox:PRMICI : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:CYTBD4um : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DM_co_c : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:CAT : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:GLUR : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:ALAR : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:SULR_2 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:NAt3pp : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:ADCS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:UGMDDS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:BIOMASS_CARB : thermo constraint NOT created

DEBUG:thermomodel_Model Exported from COBRA Toolbox:G3PAT1819Z_1 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:AGPAT160 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:AGPATACP_HDE_PALM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:AGPAT161 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:AGPATACP_OLE_HDE : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:AGPATACP_OLE_PALM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PAPA160 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PAPA_HDE_PALM : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PAPA161 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PAPA_OLE_HDE : thermo constraint NOT created
DEBUG:thermomodel_Model Exported 

DEBUG:thermomodel_Model Exported from COBRA Toolbox:DASYN182_9_12 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DASYN183_6_9_12 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DASYN183_9_12_15 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DASYN184_6_9_12_15 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DHDPS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DHORDi : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:DPPS : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EAR121y : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EAR141y : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:EAR161y : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:

DEBUG:thermomodel_Model Exported from COBRA Toolbox:HACD4 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HACD5 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HACD6 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HACD7 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HACD8 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HADPCOADH3 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HBZOPT : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HEPT1 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HEPT2 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HG2abcpp : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:HISTD : thermo constraint NOT created
D

DEBUG:thermomodel_Model Exported from COBRA Toolbox:OCOAT1 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:OOR3r : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:OPHHX : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:OXOAEL : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:OXPTNDH : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PACCOAL3 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PC : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PEPCK_re : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PGK_1 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PHAPC100 : thermo constraint NOT created
DEBUG:thermomodel_Model Exported from COBRA Toolbox:PHAPC120 : thermo constraint NOT creat

Streaming output truncated to the last 5000 lines.
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating thermo variables for starch[e]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: LC_starch[e]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating thermo variables for zcarote[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: LC_zcarote[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating thermo variables for 1h12dhl[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: LC_1h12dhl[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating thermo variables for 34dhardv[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: LC_34dhardv[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating thermo variables for 34dhrdv[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:Added variable: LC_34dhrdv[c]
DEBUG:thermomodel_Model Exported from COBRA Toolbox:generating t

In [ ]:


# --------------------------------------
# 3. MANUAL SEED-ID ANNOTATION (COFACTORS ONLY)
# --------------------------------------
SEED_MAP1 = {
    "h[c]":     "cpd00067",
    "nadp[c]":  "cpd00005",
    "nadph[c]": "cpd00006",
    "coa[c]":   "cpd00010"
}

for met_id, seed_id in SEED_MAP1.items():
    met = tmodel.metabolites.get_by_id(met_id)
    met.annotation["seed_id"] = seed_id


conc_bounds = {
    "nadph[c]": (1e-6, 1e-3),
    "nadp[c]":  (1e-6, 1e-3),
    "h[c]":     (1e-8, 1e-6),
    "coa[c]":   (1e-6, 1e-3),
}

for met_id, bounds in conc_bounds.items():
    met = tmodel.metabolites.get_by_id(met_id)
    met.concentration = bounds

# --------------------------------------
# 5. FORCE THERMODYNAMIC FEASIBILITY
#    Acetoacetyl-CoA reductase (AACOAR)
# --------------------------------------
# Reaction: aacoa + nadph + H -> 3hbcoa + nadp
# Literature ΔG°′ ≈ −25 to −30 kJ/mol
# We enforce directionality conservatively


from pytfa.optim.variables import DeltaG

rxn = tmodel.reactions.get_by_id("AACOAR_syn")
dg = tmodel.variables[DeltaG][rxn]

# Set thermodynamic directionality
dg.upper_bound = -1.0   # kJ/mol
dg.lower_bound = -100.0

# --------------------------------------
# 6. OPTIONAL: OBJECTIVE (PHB / growth)
# --------------------------------------
# Example: maximize PHB precursor
# tmodel.objective = "PHB_SYN"

# --------------------------------------
# 7. RUN TFA OPTIMIZATION
# --------------------------------------
solution = tmodel.optimize()

dg_value = solution.raw[dg.name]
print("ΔG_AACOAR (kJ/mol):", dg_value)


# --------------------------------------
# 8. REPORT RESULTS
# --------------------------------------
print("TFA status:", solution.status)
print("Objective value:", solution.objective_value)

print("\n--- AACOAR thermodynamics ---")
print("Flux:", solution.fluxes["AACOAR_syn"])
print("ΔG (kJ/mol):", solution.raw[tmodel.reactions.AACOAR_syn.delta_g.name])

# Optional: check PHB precursor
if "3hbcoa__R[c]" in solution.fluxes:
    print("\n3HB-CoA flux:", solution.fluxes["3hbcoa__R[c]"])


KeyError: <class 'pytfa.optim.variables.DeltaG'>

In [ ]:
from pytfa.optim.variables import DeltaG

phb_rxns = [
    "AACOAR_syn",
    "PHBS_syn",
    "ACACT1r",
    "ACACCT",
]

# Force thermo mode
tmodel.convert(add_thermo=True)

for r_id in phb_rxns:
    rxn = tmodel.reactions.get_by_id(r_id)

    dg = tmodel.variables[DeltaG][rxn]
    dg.lower_bound = -100
    dg.upper_bound = -5   # irreversible forward


TypeError: ThermoModel.convert() got an unexpected keyword argument 'add_thermo'

In [ ]:
def has_dGr_variable(tmodel, rxn_id):
    return any(v.name == f"DG_r_{rxn_id}" for v in tmodel.variables)

for rxn_id in ["ACACT1", "ACCOAC", "PHBS_syn"]:
    print(rxn_id, has_dGr_variable(tmodel, rxn_id))

def thermo_constraints_for_rxn(tmodel, rxn_id):
    return [
        c.name for c in tmodel.constraints
        if rxn_id in c.name and "thermo" in c.name.lower()
    ]

for rxn_id in ["ACACT1", "ACCOAC", "PHBS_syn"]:
    cons = thermo_constraints_for_rxn(tmodel, rxn_id)
    print(rxn_id, cons)


rows = []
for rxn_id in ["ACACT1", "ACCOAC", "PHBS_syn"]:
    rows.append({
        "reaction": rxn_id,
        "has_DG": has_dGr_variable(tmodel, rxn_id),
        "n_thermo_constraints": len(thermo_constraints_for_rxn(tmodel, rxn_id))
    })

pd.DataFrame(rows)


ACACT1 False
ACCOAC False
PHBS_syn False
ACACT1 []
ACCOAC []
PHBS_syn []


,reaction,has_DG,n_thermo_constraints
0,ACACT1,False,0
1,ACCOAC,False,0
2,PHBS_syn,False,0


In [ ]:
thermo_rxns = []

for rxn in tmodel.reactions:
    rxn_id = rxn.id

    # 1) check ΔG variable
    has_dg = f"DG_r_{rxn_id}" in [v.name for v in tmodel.variables]

    if not has_dg:
        continue

    # 2) check thermo constraints
    thermo_cons = [
        c for c in tmodel.constraints
        if rxn_id in c.name and "thermo" in c.name.lower()
    ]

    if len(thermo_cons) > 0:
        thermo_rxns.append(rxn_id)

print(f"Number of thermo-constrained reactions: {len(thermo_rxns)}")
thermo_rxns[:20]


Number of thermo-constrained reactions: 0


[]

In [ ]:
rxn = tmodel.reactions.get_by_id('PHBS_syn')
for met in rxn.metabolites:
    print(met.id, getattr(met, "seed_id", "no seed_id"))


3hbcoa__R[c] C12345
phbg[c] C23456
coa[c] C00010
PHB[c] C34567


In [ ]:
# List all variables (variable objects)
for var in tmodel.variables:
    print(var)


In [ ]:
rxn = tmodel.reactions.get_by_id('PHBS_syn')

if hasattr(rxn, 'deltaG_r') and rxn.deltaG_r in tmodel.variables:
    print(f"{rxn.id} now has a thermodynamic constraint (ΔG applied).")
else:
    print(f"{rxn.id} still does NOT have a thermodynamic constraint.")


PHBS_syn still does NOT have a thermodynamic constraint.


## Map to ModelSEED Database

*   https://modelseed.org/biochem/compounds/cpd00805

*   https://academic.oup.com/nar/article/49/D1/D575/5912569

*   https://github.com/ModelSEED/ModelSEEDpy/tree/dev/tests/biochem

*   https://github.com/ModelSEED/ModelSEEDDatabase?utm_source=chatgpt.com




In [ ]:
import json
from pathlib import Path

db_path = Path("ModelSEEDDatabase/Biochemistry/compounds.json")
if not db_path.exists():
    raise FileNotFoundError(f"File not found: {db_path}")

with open(db_path) as f:
    compounds = json.load(f)


FileNotFoundError: File not found: ModelSEEDDatabase/Biochemistry/compounds.json

In [ ]:
import json
from pathlib import Path

# Load ModelSEED compounds
with open("ModelSEEDDatabase/Biochemistry/compounds.json") as f:
    compounds = json.load(f)  # dict keyed by 'cpdXXXX'
    print(compounds)


import pandas as pd

# Load your GEM metabolites
gem_mets = [
    '13dpg_c', '2pg_c', '3pg_c', '6pgc_c', '6pgl_c', 'ac_c', 'ac_e',
    'acald_c', 'acald_e', 'accoa_c', 'acon_C_c', 'actp_c', 'adp_c',
    'akg_c', 'akg_e', 'amp_c', 'atp_c', 'cit_c', 'co2_c', 'co2_e'
]

# Load ModelSEED thermodynamic database (JSON or CSV)
thermobase = pd.read_json(db_path)  # replace with your thermodb path
# Example columns: 'id', 'name', 'formula', 'synonyms'

def map_to_modelseed(gem_mets, gem_met_info, thermobase):
    """
    gem_met_info: dict {met_id: {'name': ..., 'formula': ...}}
    thermobase: pandas DataFrame of ModelSEED compounds
    """
    mapping = {}
    unmatched = []
    for met in gem_mets:
        info = gem_met_info.get(met, {})
        formula = info.get('formula', None)
        name = info.get('name', None)

        # Match by formula first
        candidates = thermobase[thermobase['formula'] == formula]
        if len(candidates) == 1:
            mapping[met] = candidates['id'].values[0]
            continue

        # Match by name / synonym
        candidates = thermobase[
            thermobase['name'].str.lower() == name.lower() |
            thermobase['synonyms'].apply(lambda x: name.lower() in [s.lower() for s in x])
        ]
        if len(candidates) >= 1:
            mapping[met] = candidates['id'].values[0]
        else:
            unmatched.append(met)

    return mapping, unmatched

# Example usage:
# gem_met_info = {met.id: {'name': met.name, 'formula': met.formula} for met in cobra_model.metabolites}
# mapping, unmatched = map_to_modelseed(gem_mets, gem_met_info, thermobase)


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
print("Number of metabolites with ΔG° info:", len(tmodel.metabolites))
print("Number of reactions with ΔG° info:", len(tmodel.reactions))


Number of metabolites with ΔG° info: 72
Number of reactions with ΔG° info: 95


In [ ]:
tmodel.prepare()  # sets up thermodynamic variables and constraints
tmodel.convert()  # adds thermodynamic constraints to the COBRA model


2025-12-02 18:56:22,544 - thermomodel_None - INFO - # Model preparation starting...
INFO:thermomodel_None:# Model preparation starting...
DEBUG:thermomodel_None:Metabolite 13dpg_c (3-Phospho-D-glyceroyl phosphate) has no seed_id
DEBUG:thermomodel_None:Metabolite 2pg_c (D-Glycerate 2-phosphate) has no seed_id
DEBUG:thermomodel_None:Metabolite 3pg_c (3-Phospho-D-glycerate) has no seed_id
DEBUG:thermomodel_None:Metabolite 6pgc_c (6-Phospho-D-gluconate) has no seed_id
DEBUG:thermomodel_None:Metabolite 6pgl_c (6-phospho-D-glucono-1,5-lactone) has no seed_id
DEBUG:thermomodel_None:Metabolite ac_c (Acetate) has no seed_id
DEBUG:thermomodel_None:Metabolite ac_e (Acetate) has no seed_id
DEBUG:thermomodel_None:Metabolite acald_c (Acetaldehyde) has no seed_id
DEBUG:thermomodel_None:Metabolite acald_e (Acetaldehyde) has no seed_id
DEBUG:thermomodel_None:Metabolite accoa_c (Acetyl-CoA) has no seed_id
DEBUG:thermomodel_None:Metabolite acon_C_c (cis-Aconitate) has no seed_id
DEBUG:thermomodel_None:Me